
# Stage 0 — Setup & Utilities

## What this cell does:
 - Imports libraries
 - Lets you point to your graphs dir
 - Defines helpers for parsing filenames, reading GraphML edges, computing CSA_mm2
 - Adds region labels and pretty display utilities
 - Notes on actual GraphML structure (based on inspection)


In [1]:
import re
from pathlib import Path
import numpy as np
import pandas as pd
import networkx as nx

GRAPH_DIR = Path(r"C:\Users\ilinc\OneDrive\Desktop\GraphAnalysis\GraphsCompleteAnalysis\graphs_complete_cleaned\graphs_complete_cleaned\_final_basic_clean\_matched_outputs")

OUT_DIR = Path(r"C:\Users\ilinc\OneDrive\Desktop\GraphAnalysis\GraphsCompleteAnalysis\FinalStatisticalAnalysisResults")
OUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_rows", 50)
pd.set_option("display.max_columns", 0)
pd.set_option("display.width", 160)

FU_DIR_RE = re.compile(r"^FU(?P<fu_idx>\d+)(?:-M(?P<months>[\d\.]+))?$", re.IGNORECASE)

def parse_tp_from_dirname(tp_dirname: str):
    d = str(tp_dirname).strip()
    if d.upper() == "BASELINE":
        return dict(tp="BL", tpraw="BASELINE-M0", fu_order=0.0, months=0.0)
    m = FU_DIR_RE.match(d)
    if not m:
        return None
    fu_idx = float(m.group("fu_idx"))
    months = float(m.group("months")) if m.group("months") is not None else np.nan
    tpraw = f"FU{int(fu_idx)}" + (f"-M{months}" if np.isfinite(months) else "")
    return dict(tp="FU", tpraw=tpraw, fu_order=fu_idx, months=months)

def _to_float(x):
    try:
        return float(x)
    except Exception:
        return np.nan

def _to_bool_from_numeric(x):
    s = str(x).strip()
    if s == "":
        return np.nan
    try:
        return bool(int(s))
    except Exception:
        if s.lower() in ["true","t","yes","y"]:
            return True
        if s.lower() in ["false","f","no","n"]:
            return False
        return np.nan

def pick_existing_file(*candidates: Path):
    for p in candidates:
        if p is None:
            continue
        if isinstance(p, (str, Path)):
            p = Path(p)
        if p.exists() and p.is_file():
            return p
    return None

def read_graph_edges(path: Path) -> pd.DataFrame:
    G = nx.read_graphml(path)
    rows = []
    for (_, _, attr) in G.edges(data=True):
        volume = _to_float(attr.get("volume", np.nan))
        length = _to_float(attr.get("length", np.nan))
        radius_avg = _to_float(attr.get("radius_avg", np.nan))

        if np.isfinite(radius_avg) and radius_avg > 0:
            csa = np.pi * (radius_avg ** 2)
        elif np.isfinite(volume) and np.isfinite(length) and length > 0:
            csa = volume / length
        else:
            csa = np.nan

        rows.append({
            "volume_mm3": volume,
            "surface_area": _to_float(attr.get("surface_area", np.nan)),
            "length": length,
            "tortuosity": _to_float(attr.get("tortuosity", np.nan)),
            "radius_avg": radius_avg,
            "radius_max": _to_float(attr.get("radius_max", np.nan)),
            "radius_min": _to_float(attr.get("radius_min", np.nan)),
            "radius_SD":  _to_float(attr.get("radius_SD",  np.nan)),
            "vis_radius": _to_float(attr.get("vis_radius", np.nan)),
            "CSA_mm2": csa,

            "lobe": str(attr.get("lobe")) if attr.get("lobe") is not None else None,
            "side": str(attr.get("side")) if attr.get("side") is not None else None,

            "is_ipsilateral": _to_bool_from_numeric(attr.get("is_ipsilateral", "")),
            "is_in_tumor_lobe": _to_bool_from_numeric(attr.get("is_in_tumor_lobe", "")),
            "dose_gy": _to_float(attr.get("dose_gy", np.nan)),
            "multiplicity": _to_float(attr.get("multiplicity", np.nan)),
        })
    return pd.DataFrame(rows)

def add_region_cols(df: pd.DataFrame) -> pd.DataFrame:
    side_str = df["side"].astype(str).str.strip().str.lower()
    central_flag = side_str.eq("central")
    df = df.assign(central=central_flag)

    def region_of_row(r):
        if bool(r["central"]):
            return "central"
        if pd.notna(r["is_ipsilateral"]):
            return "ipsi" if bool(r["is_ipsilateral"]) else "contra"
        return "unknown"

    df["region"] = df.apply(region_of_row, axis=1)
    return df

def discover_graph_files(root: Path) -> pd.DataFrame:
    records = []

    roots_to_scan = [root]
    rest_root = root / "rest"
    if rest_root.exists():
        roots_to_scan.append(rest_root)

    for base_root in roots_to_scan:
        for patient_dir in sorted(base_root.glob("P*")):
            if not patient_dir.is_dir():
                continue
            patient = patient_dir.name

            for tree_dir in [patient_dir / "Artery", patient_dir / "Vein"]:
                if not tree_dir.exists():
                    continue
                tree = tree_dir.name

                meta = parse_tp_from_dirname("BASELINE")
                bl_file = pick_existing_file(
                    tree_dir / "BASELINE" / "sanitize" / "central_sanitized.graphml",
                    tree_dir / "BASELINE" / "sanitize" / "central_sanitized",
                )
                if bl_file is not None:
                    records.append({
                        "patient": patient,
                        "tree": tree,
                        "tp": meta["tp"],
                        "tpraw": meta["tpraw"],
                        "fu_order": meta["fu_order"],
                        "months": meta["months"],
                        "path": str(bl_file),
                        "source": "BASELINE/sanitize/central_sanitized",
                        "root_branch": "rest" if base_root.name == "rest" else "main",
                    })

                for tp_dir in sorted(tree_dir.glob("FU*")):
                    if not tp_dir.is_dir():
                        continue
                    meta = parse_tp_from_dirname(tp_dir.name)
                    if meta is None:
                        continue

                    fu_file = pick_existing_file(
                        tp_dir / "sanitize" / "central_sanitized.graphml",
                        tp_dir / "sanitize" / "central_sanitized",
                        # fallback
                        tp_dir / "prematch_rigid" / "FU_sanitized_shifted.graphml",
                    )

                    if fu_file is None:
                        continue

                    source = "FU/sanitize/central_sanitized" if "sanitize" in str(fu_file) else "FU/prematch_rigid/FU_sanitized_shifted"
                    records.append({
                        "patient": patient,
                        "tree": tree,
                        "tp": meta["tp"],
                        "tpraw": meta["tpraw"],
                        "fu_order": meta["fu_order"],
                        "months": meta["months"],
                        "path": str(fu_file),
                        "source": source,
                        "root_branch": "rest" if base_root.name == "rest" else "main",
                    })

    df = pd.DataFrame.from_records(records)
    if df.empty:
        return df

    df["fu_order"] = pd.to_numeric(df["fu_order"], errors="coerce")
    df["months"] = pd.to_numeric(df["months"], errors="coerce")
    df = df.sort_values(["patient", "tree", "fu_order", "months", "tpraw"]).reset_index(drop=True)
    return df

def display_head(df, title, n=10):
    print(title)
    print("-" * len(title))
    display(df.head(n))

files = discover_graph_files(GRAPH_DIR)
display_head(files, "Discovered graph files", n=15) 

Discovered graph files
----------------------


,patient,tree,tp,tpraw,fu_order,months,path,source,root_branch
0,P104,Artery,BL,BASELINE-M0,0.0,0.0,C:\Users\ilinc\OneDrive\Desktop\GraphAnalysis\...,BASELINE/sanitize/central_sanitized,rest
1,P104,Artery,FU,FU1-M5.0,1.0,5.0,C:\Users\ilinc\OneDrive\Desktop\GraphAnalysis\...,FU/sanitize/central_sanitized,rest
2,P104,Artery,FU,FU2-M15.0,2.0,15.0,C:\Users\ilinc\OneDrive\Desktop\GraphAnalysis\...,FU/sanitize/central_sanitized,rest
3,P104,Artery,FU,FU3-M27.0,3.0,27.0,C:\Users\ilinc\OneDrive\Desktop\GraphAnalysis\...,FU/sanitize/central_sanitized,rest
4,P104,Artery,FU,FU4-M41.0,4.0,41.0,C:\Users\ilinc\OneDrive\Desktop\GraphAnalysis\...,FU/sanitize/central_sanitized,rest
5,P104,Vein,BL,BASELINE-M0,0.0,0.0,C:\Users\ilinc\OneDrive\Desktop\GraphAnalysis\...,BASELINE/sanitize/central_sanitized,rest
6,P104,Vein,FU,FU1-M5.0,1.0,5.0,C:\Users\ilinc\OneDrive\Desktop\GraphAnalysis\...,FU/sanitize/central_sanitized,rest
7,P104,Vein,FU,FU2-M15.0,2.0,15.0,C:\Users\ilinc\OneDrive\Desktop\GraphAnalysis\...,FU/sanitize/central_sanitized,rest
8,P104,Vein,FU,FU3-M27.0,3.0,27.0,C:\Users\ilinc\OneDrive\Desktop\GraphAnalysis\...,FU/sanitize/central_sanitized,rest
9,P104,Vein,FU,FU4-M41.0,4.0,41.0,C:\Users\ilinc\OneDrive\Desktop\GraphAnalysis\...,FU/sanitize/central_sanitized,rest


# Stage A — Parse graphs and extract all relevant data

## What this cell does:

 - Walks GRAPH_DIR, parses filenames for patient/timepoint/tree
 - Sets BL months=0 and fu_order=0 (no NaNs at BL)
 - Picks FU_earliest / FU_latest per patient (by months if present, else FU order)
 - Reads each GraphML, extracts edge attributes 
 - Computes CSA_mm2 =  𝜋*radius_avg^(2)
 - Adds region labels (whole/ipsi/contra/central/lobes)
 - Saves edges_tidy_stageA.csv
 - Prints a per-patient graphs processed table & selection overview
 - Shows heads of files_df and edges_df for sanity check

In [2]:
from collections import defaultdict
from pathlib import Path
import pandas as pd
import numpy as np

graph_dir = Path(GRAPH_DIR)
#print(graph_dir)
assert graph_dir.exists(), f"Graph directory not found: {GRAPH_DIR}"

files_df = discover_graph_files(graph_dir).copy()
files_df["path"] = files_df["path"].astype(str)

assert not files_df.empty, "No recognized graph files"

def _priority_row(r):
    rb = str(r.get("root_branch", "")).lower()
    src = str(r.get("source", "")).lower()
    p_rest = 1 if rb == "rest" else 0
    p_sanitize = 1 if "sanitize/central_sanitized" in src else 0
    p_has_months = 1 if pd.notna(r.get("months", np.nan)) else 0
    return (p_rest, p_sanitize, p_has_months)

if "root_branch" in files_df.columns or "source" in files_df.columns:
    pr = files_df.apply(_priority_row, axis=1, result_type="expand")
    pr.columns = ["p_rest", "p_sanitize", "p_has_months"]
    files_df = pd.concat([files_df, pr], axis=1)
    files_df = (
        files_df
        .sort_values(["patient","tree","tpraw","p_rest","p_sanitize","p_has_months"], ascending=[True,True,True,False,False,False])
        .drop_duplicates(subset=["patient","tree","tpraw"], keep="first")
        .reset_index(drop=True)
    )


files_df["tp_sel"] = "FU_other"
files_df.loc[files_df["tp"] == "BL", "tp_sel"] = "BL"

def mark_earliest_latest(g: pd.DataFrame) -> pd.DataFrame:
    g = g.copy()
    fu = g[g["tp"] == "FU"].copy()
    if fu.empty:
        return g
    if fu["months"].notna().any():
        fu2 = fu[fu["months"].notna()].copy()
        if not fu2.empty:
            i_ear = fu2["months"].astype(float).idxmin()
            i_lat = fu2["months"].astype(float).idxmax()
            g.loc[i_ear, "tp_sel"] = "FU_earliest"
            g.loc[i_lat, "tp_sel"] = "FU_latest"
            return g

    fu3 = fu[fu["fu_order"].notna()].copy()
    if not fu3.empty:
        i_ear = fu3["fu_order"].astype(float).idxmin()
        i_lat = fu3["fu_order"].astype(float).idxmax()
        g.loc[i_ear, "tp_sel"] = "FU_earliest"
        g.loc[i_lat, "tp_sel"] = "FU_latest"
    return g

files_df = (
    files_df
    .groupby(["patient","tree"], group_keys=False)
    .apply(mark_earliest_latest)
    .reset_index(drop=True)
)

edge_tables = []
skipped = 0

for _, r in files_df.iterrows():
    try:
        df = read_graph_edges(Path(r["path"]))
        if df.empty:
            continue

        df["patient"]  = r["patient"]
        df["tp"]       = r["tp"]
        df["tpraw"]    = r["tpraw"]
        df["months"]   = r["months"]
        df["fu_order"] = r["fu_order"]
        df["tp_sel"]   = r["tp_sel"]
        df["tree"]     = r["tree"]   
        if "source" in r.index:      
            df["source"] = r["source"]
        if "root_branch" in r.index:
            df["root_branch"] = r["root_branch"]

        edge_tables.append(df)

    except Exception as e:
        print(f"Failed to read {r['path']}: {e}")
        skipped += 1

edges_df = pd.concat(edge_tables, ignore_index=True) if edge_tables else pd.DataFrame()
edges_df = add_region_cols(edges_df)

edges_df["vol_AL_mm3"] = edges_df["CSA_mm2"].astype(float) * edges_df["length"].astype(float)

edges_df.to_csv(OUT_DIR / "all_edges_tidy_stageA.csv", index=False)
print("Stage A complete")

per_patient_counts = (
    files_df
      .pivot_table(index=["patient","tree"], columns="tp_sel", values="path", aggfunc="count", fill_value=0)
      .reset_index()
      .rename_axis(None, axis=1)
      .sort_values(["patient","tree"])
)

sel_overview = (
    files_df.loc[files_df["tp_sel"].isin(["BL","FU_earliest","FU_latest"]),
                 ["patient","tree","tp_sel","tpraw","months","fu_order","path"]]
            .drop_duplicates()
            .sort_values(["patient","tree","tp_sel"])
)

total_edges = len(edges_df)
n_files = len(files_df)
n_patients = edges_df["patient"].nunique() if not edges_df.empty else 0

print(f" - Parsed edges: {total_edges:,} from {n_files} graphs, {n_patients} patients (skipped files={skipped})")
print(" - Graphs processed per patient/tree:")
display(per_patient_counts)

print(" - Selected timepoints per patient/tree (BL / earliest / latest):")
display(sel_overview)

display_head(files_df.sort_values(["patient","tree","tp_sel"]), "Files parsed (first rows)")
display_head(edges_df, "edges_df (first rows)") 

Stage A complete
 - Parsed edges: 639,676 from 225 graphs, 33 patients (skipped files=0)
 - Graphs processed per patient/tree:


,patient,tree,BL,FU_earliest,FU_latest,FU_other
0,P104,Artery,1,1,1,2
1,P104,Vein,1,1,1,2
2,P105,Artery,1,1,1,3
3,P105,Vein,1,1,1,3
4,P106,Artery,1,1,1,0
...,...,...,...,...,...,...
61,P82,Vein,1,1,1,0
62,P87,Artery,1,1,1,2
63,P87,Vein,1,1,1,2
64,P96,Artery,1,1,1,3


 - Selected timepoints per patient/tree (BL / earliest / latest):


,patient,tree,tp_sel,tpraw,months,fu_order,path
0,P104,Artery,BL,BASELINE-M0,0.0,0.0,C:\Users\ilinc\OneDrive\Desktop\GraphAnalysis\...
1,P104,Artery,FU_earliest,FU1-M5.0,5.0,1.0,C:\Users\ilinc\OneDrive\Desktop\GraphAnalysis\...
4,P104,Artery,FU_latest,FU4-M41.0,41.0,4.0,C:\Users\ilinc\OneDrive\Desktop\GraphAnalysis\...
5,P104,Vein,BL,BASELINE-M0,0.0,0.0,C:\Users\ilinc\OneDrive\Desktop\GraphAnalysis\...
6,P104,Vein,FU_earliest,FU1-M5.0,5.0,1.0,C:\Users\ilinc\OneDrive\Desktop\GraphAnalysis\...
...,...,...,...,...,...,...,...
214,P96,Artery,FU_earliest,FU1-M5.0,5.0,1.0,C:\Users\ilinc\OneDrive\Desktop\GraphAnalysis\...
218,P96,Artery,FU_latest,FU5-M31.0,31.0,5.0,C:\Users\ilinc\OneDrive\Desktop\GraphAnalysis\...
219,P96,Vein,BL,BASELINE-M0,0.0,0.0,C:\Users\ilinc\OneDrive\Desktop\GraphAnalysis\...
220,P96,Vein,FU_earliest,FU1-M5.0,5.0,1.0,C:\Users\ilinc\OneDrive\Desktop\GraphAnalysis\...


Files parsed (first rows)
-------------------------


,patient,tree,tp,tpraw,fu_order,months,path,source,root_branch,p_rest,p_sanitize,p_has_months,tp_sel
0,P104,Artery,BL,BASELINE-M0,0.0,0.0,C:\Users\ilinc\OneDrive\Desktop\GraphAnalysis\...,BASELINE/sanitize/central_sanitized,rest,1,1,1,BL
1,P104,Artery,FU,FU1-M5.0,1.0,5.0,C:\Users\ilinc\OneDrive\Desktop\GraphAnalysis\...,FU/sanitize/central_sanitized,rest,1,1,1,FU_earliest
4,P104,Artery,FU,FU4-M41.0,4.0,41.0,C:\Users\ilinc\OneDrive\Desktop\GraphAnalysis\...,FU/sanitize/central_sanitized,rest,1,1,1,FU_latest
2,P104,Artery,FU,FU2-M15.0,2.0,15.0,C:\Users\ilinc\OneDrive\Desktop\GraphAnalysis\...,FU/sanitize/central_sanitized,rest,1,1,1,FU_other
3,P104,Artery,FU,FU3-M27.0,3.0,27.0,C:\Users\ilinc\OneDrive\Desktop\GraphAnalysis\...,FU/sanitize/central_sanitized,rest,1,1,1,FU_other
5,P104,Vein,BL,BASELINE-M0,0.0,0.0,C:\Users\ilinc\OneDrive\Desktop\GraphAnalysis\...,BASELINE/sanitize/central_sanitized,rest,1,1,1,BL
6,P104,Vein,FU,FU1-M5.0,1.0,5.0,C:\Users\ilinc\OneDrive\Desktop\GraphAnalysis\...,FU/sanitize/central_sanitized,rest,1,1,1,FU_earliest
9,P104,Vein,FU,FU4-M41.0,4.0,41.0,C:\Users\ilinc\OneDrive\Desktop\GraphAnalysis\...,FU/sanitize/central_sanitized,rest,1,1,1,FU_latest
7,P104,Vein,FU,FU2-M15.0,2.0,15.0,C:\Users\ilinc\OneDrive\Desktop\GraphAnalysis\...,FU/sanitize/central_sanitized,rest,1,1,1,FU_other
8,P104,Vein,FU,FU3-M27.0,3.0,27.0,C:\Users\ilinc\OneDrive\Desktop\GraphAnalysis\...,FU/sanitize/central_sanitized,rest,1,1,1,FU_other


edges_df (first rows)
---------------------


,volume_mm3,surface_area,length,tortuosity,radius_avg,radius_max,radius_min,radius_SD,vis_radius,CSA_mm2,lobe,side,is_ipsilateral,is_in_tumor_lobe,dose_gy,multiplicity,patient,tp,tpraw,months,fu_order,tp_sel,tree,source,root_branch,central,region,vol_AL_mm3
0,26.747610,59.672073,10.593694,1.054112,0.896487,1.493673,0.500000,0.337517,0.896487,2.524862,5,Right,False,False,0.167524,1.0,P104,BL,BASELINE-M0,0.0,0.0,BL,Artery,BASELINE/sanitize/central_sanitized,rest,False,contra,26.747610
1,82.787579,150.090718,21.653725,1.025334,1.103167,1.652591,0.500000,0.343335,1.103167,3.823249,5,Right,False,False,0.317428,1.0,P104,BL,BASELINE-M0,0.0,0.0,BL,Artery,BASELINE/sanitize/central_sanitized,rest,False,contra,82.787579
2,23.361864,53.619197,9.793169,1.010088,0.871399,1.185660,0.500000,0.278458,0.871399,2.385527,5,Right,False,False,0.254737,1.0,P104,BL,BASELINE-M0,0.0,0.0,BL,Artery,BASELINE/sanitize/central_sanitized,rest,False,contra,23.361864
3,23.652384,47.532973,7.601603,1.006857,0.995199,1.185660,0.500000,0.277316,0.995199,3.111500,5,Right,False,False,0.255121,1.0,P104,BL,BASELINE-M0,0.0,0.0,BL,Artery,BASELINE/sanitize/central_sanitized,rest,False,contra,23.652384
4,24.055778,43.364227,6.220624,1.009118,1.109476,1.414214,0.728553,0.285056,1.109476,3.867101,5,Right,False,False,0.169115,1.0,P104,BL,BASELINE-M0,0.0,0.0,BL,Artery,BASELINE/sanitize/central_sanitized,rest,False,contra,24.055778
5,61.990669,101.664847,13.267999,1.036057,1.219510,1.493673,0.957107,0.141515,1.219510,4.672194,5,Right,False,False,0.226513,1.0,P104,BL,BASELINE-M0,0.0,0.0,BL,Artery,BASELINE/sanitize/central_sanitized,rest,False,contra,61.990669
6,46.335989,97.841206,16.440510,1.031569,0.947167,1.493673,0.500000,0.336334,0.947167,2.818403,5,Right,False,False,0.370376,1.0,P104,BL,BASELINE-M0,0.0,0.0,BL,Artery,BASELINE/sanitize/central_sanitized,rest,False,contra,46.335989
7,47.746545,85.483475,12.179025,1.076484,1.117094,1.414214,0.728553,0.178506,1.117094,3.920391,5,Right,False,False,0.225237,1.0,P104,BL,BASELINE-M0,0.0,0.0,BL,Artery,BASELINE/sanitize/central_sanitized,rest,False,contra,47.746545
8,56.399995,79.334597,8.880477,1.005516,1.421826,1.800042,1.185660,0.211680,1.421826,6.351009,5,Right,False,False,0.275577,1.0,P104,BL,BASELINE-M0,0.0,0.0,BL,Artery,BASELINE/sanitize/central_sanitized,rest,False,contra,56.399995
9,33.065711,67.627038,11.006595,1.049438,0.977884,1.185660,0.500000,0.283368,0.977884,3.004173,5,Right,False,False,0.452172,1.0,P104,BL,BASELINE-M0,0.0,0.0,BL,Artery,BASELINE/sanitize/central_sanitized,rest,False,contra,33.065711


In [3]:
import numpy as np
import pandas as pd

# tp_sel exists in files_df
if "tp_sel" not in files_df.columns:
    files_df = files_df.copy()
    files_df["tp_sel"] = "FU_other"
    files_df.loc[files_df["tp"].astype(str).str.upper().eq("BL"), "tp_sel"] = "BL"

    def mark_earliest_latest(g: pd.DataFrame) -> pd.DataFrame:
        g = g.copy()
        fu = g[g["tp"].astype(str).str.upper().eq("FU")].copy()
        if fu.empty:
            return g
        m = pd.to_numeric(fu.get("months", np.nan), errors="coerce")
        if np.isfinite(m).any():
            fu2 = fu.loc[np.isfinite(m)].copy()
            if not fu2.empty:
                i_ear = pd.to_numeric(fu2["months"], errors="coerce").idxmin()
                i_lat = pd.to_numeric(fu2["months"], errors="coerce").idxmax()
                g.loc[i_ear, "tp_sel"] = "FU_earliest"
                g.loc[i_lat, "tp_sel"] = "FU_latest"
                return g

            o = pd.to_numeric(fu.get("fu_order", np.nan), errors="coerce")
        if np.isfinite(o).any():
            fu3 = fu.loc[np.isfinite(o)].copy()
            i_ear = pd.to_numeric(fu3["fu_order"], errors="coerce").idxmin()
            i_lat = pd.to_numeric(fu3["fu_order"], errors="coerce").idxmax()
            g.loc[i_ear, "tp_sel"] = "FU_earliest"
            g.loc[i_lat, "tp_sel"] = "FU_latest"
        return g

    files_df = (
        files_df.groupby(["patient","tree"], group_keys=False)
                .apply(mark_earliest_latest)
                .reset_index(drop=True)
    )

print("tp_sel value counts:", files_df["tp_sel"].value_counts(dropna=False).to_dict())


sel_counts = (
    files_df.pivot_table(index=["patient","tree"], columns="tp_sel", values="path",
                         aggfunc="count", fill_value=0)
    .reset_index()
)

has_fu = (
    files_df.groupby(["patient","tree"])["tp"]
            .apply(lambda s: (s.astype(str).str.upper() == "FU").any())
            .reset_index(name="has_fu")
)
sel_counts = sel_counts.merge(has_fu, on=["patient","tree"], how="left")

def _bad_row(r):
    if r.get("BL", 0) != 1:
        return True
    if r["has_fu"]:
        return (r.get("FU_earliest", 0) != 1) or (r.get("FU_latest", 0) != 1)
    return False

bad_sel = sel_counts[sel_counts.apply(_bad_row, axis=1)]
print("Patient/tree with missing BL or missing earliest/latest:", len(bad_sel))
display(bad_sel.head(100))

tp_sel value counts: {'BL': 66, 'FU_other': 55, 'FU_earliest': 52, 'FU_latest': 52}
Patient/tree with missing BL or missing earliest/latest: 0


,patient,tree,BL,FU_earliest,FU_latest,FU_other,has_fu


# Stage B — Aggregate A/V and A+V metrics per patient × timepoint × region

## What this cell does:

 - TBV = sum(volume_mm3)
 - BVx (x∈{3,5,7,10}) = sum(volume_mm3 where CSA_mm2 < x)
 - Frac_BVx = BVx/TBV
 - Scopes: Artery, Vein, and A+V (only if both trees exist for that patient/timepoint)
 - Regions: whole, ipsi, contra, central, and each lobe
 - Saves: metrics_stageB_raw.csv
 - Displays summary + heads


In [4]:
import numpy as np
import pandas as pd
from pathlib import Path

BVX_THRESHOLDS = [3,5,7,10]

def has_both_trees(df_pt):
    trees = set(df_pt["tree"].dropna().unique())
    return ("Artery" in trees) and ("Vein" in trees)

def agg_metrics(df_slice):
    vol = pd.to_numeric(df_slice["vol_AL_mm3"], errors="coerce")
    A   = pd.to_numeric(df_slice["CSA_mm2"], errors="coerce")

    m_vol = np.isfinite(vol.values)
    tbv = float(np.nansum(vol.values[m_vol]))
    out = {"TBV": tbv}

    for x in BVX_THRESHOLDS:
        m = m_vol & np.isfinite(A.values) & (A.values < float(x))
        bx = float(np.nansum(vol.values[m]))
        out[f"BV{x}"] = bx
        out[f"Frac_BV{x}"] = float(bx / tbv) if tbv > 0 else np.nan

    return out

records = []
for (pid, tp_sel), df_pt in edges_df.groupby(["patient","tp_sel"]):
    if tp_sel not in {"BL","FU_earliest","FU_latest"}:
        continue


    lobes_present = (
        df_pt["lobe"]
        .dropna()
        .astype(str)
        .map(lambda s: s.strip())
    )
    lobes_present = sorted([lv for lv in lobes_present.unique().tolist() if lv not in {"", "0", "None", "nan", "NaN"}])

    regions = ["whole", "ipsi", "contra", "central"] + [f"lobe={lv}" for lv in lobes_present]

    for scope in ["Artery","Vein","A+V"]:
        if scope == "A+V":
            if not has_both_trees(df_pt):
                continue
            sub = df_pt
        else:
            sub = df_pt[df_pt["tree"] == scope]

        if sub.empty:
            continue

        for reg in regions:
            if reg == "whole":
                s = sub
            elif reg in {"ipsi","contra","central"}:
                s = sub[sub["region"] == reg]
            elif reg.startswith("lobe="):
                lv = reg.split("=", 1)[1]
                s = sub[sub["lobe"].astype(str).str.strip() == lv]
            else:
                continue

            if s.empty:
                continue

            met = agg_metrics(s)
            row = {"patient": pid, "tp_sel": tp_sel, "scope": scope, "region": reg}
            row.update(met)
            records.append(row)

metrics = pd.DataFrame.from_records(records)
assert not metrics.empty, "Stage B: no aggregated metrics produced."

out_path_B = OUT_DIR / "metrics_stageB_raw.csv"
metrics.to_csv(out_path_B, index=False)

print("Stage B complete")
#print(f" - Rows aggregated: {len(metrics):,}")
#print(f" - Scopes present: {sorted(metrics['scope'].unique())}")
#print(f" - Example regions: {sorted(metrics['region'].unique())[:12]} ...")

display_head(metrics.sort_values(['patient','tp_sel','scope','region']), "metrics (first rows)")


Stage B complete
metrics (first rows)
--------------------


,patient,tp_sel,scope,region,TBV,BV3,Frac_BV3,BV5,Frac_BV5,BV7,Frac_BV7,BV10,Frac_BV10
21,P104,BL,A+V,central,220663.101638,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,66.739679,0.000302
20,P104,BL,A+V,contra,210383.955954,8468.036075,0.040250,52114.531998,0.247712,88401.360453,0.420191,118802.918964,0.564696
19,P104,BL,A+V,ipsi,204330.615178,7939.721448,0.038857,44047.538955,0.215570,76114.747495,0.372508,102837.981854,0.503292
22,P104,BL,A+V,lobe=1,101851.622173,5173.051116,0.050790,26834.381483,0.263465,43798.400573,0.430022,57744.326291,0.566946
23,P104,BL,A+V,lobe=2,102478.993005,2766.670332,0.026997,17213.157472,0.167968,32316.346922,0.315346,45093.655563,0.440028
24,P104,BL,A+V,lobe=3,67979.926219,4367.358879,0.064245,21019.966161,0.309208,32857.047715,0.483335,42414.045907,0.623920
25,P104,BL,A+V,lobe=4,33689.322617,1089.071289,0.032327,7326.762963,0.217480,14098.225293,0.418478,19275.320429,0.572149
26,P104,BL,A+V,lobe=5,108714.707118,3011.605907,0.027702,23767.802874,0.218625,41446.087445,0.381237,57113.552628,0.525353
18,P104,BL,A+V,whole,635377.672770,16407.757523,0.025824,96162.070953,0.151346,164516.107948,0.258926,221707.640498,0.348938
3,P104,BL,Artery,central,110240.505124,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,66.739679,0.000605


# Stage C — Normalize metrics by Baseline-Contralateral (whole lung) per patient×scope

 - TBV_norm = TBV / TBV_contra_BL
 - BVx_norm = BVx / BVx_contra_BL
 - (BVx/TBV)_norm = (BVx/TBV) / (BVx/TBV)_contra_BL
 - Also derive tumor_lobe map from BL edges (exclude central)
 
## Saves:
 - metrics_stageB_norm.csv
 - tumor_lobe_map.csv
 - tp_selection_stageB.csv

In [5]:
import numpy as np
import pandas as pd
from pathlib import Path

OUT_DIR = Path(r"C:\Users\ilinc\OneDrive\Desktop\GraphAnalysis\GraphsCompleteAnalysis\FinalStatisticalAnalysisResults\PostResults")
ALL_METRICS_CSV = OUT_DIR / "all_metrics_stageB_raw.csv"
ALL_EDGES_CSV   = OUT_DIR / "all_edges_tidy_stageA.csv"   

assert ALL_METRICS_CSV.exists(), f"Missing: {ALL_METRICS_CSV}"
assert ALL_EDGES_CSV.exists(),   f"Missing: {ALL_EDGES_CSV}"

metrics  = pd.read_csv(ALL_METRICS_CSV)
edges_df = pd.read_csv(ALL_EDGES_CSV)

for c in ["patient","tp_sel","scope","region"]:
    if c in metrics.columns:
        metrics[c] = metrics[c].astype(str)

for c in ["patient","tp_sel","tp","tpraw","lobe"]:
    if c in edges_df.columns:
        edges_df[c] = edges_df[c].astype(str)


def baseline_contra_scalers(df_metrics, pid, scope):
    base_contra = df_metrics[
        (df_metrics["patient"] == pid) &
        (df_metrics["tp_sel"] == "BL") &
        (df_metrics["scope"]  == scope) &
        (df_metrics["region"] == "contra")
    ]
    if base_contra.empty:
        base_whole = df_metrics[
            (df_metrics["patient"] == pid) &
            (df_metrics["tp_sel"] == "BL") &
            (df_metrics["scope"]  == scope) &
            (df_metrics["region"] == "whole")
        ]
        if base_whole.empty:
            return None
        base_contra = base_whole

    sc = {"TBV": float(base_contra["TBV"].values[0])}
    for x in BVX_THRESHOLDS:
        sc[f"BV{x}"]       = float(base_contra[f"BV{x}"].values[0])
        sc[f"Frac_BV{x}"]  = float(base_contra[f"Frac_BV{x}"].values[0])
    return sc

norm_rows = []
for (pid, scope), g in metrics.groupby(["patient", "scope"]):
    sc = baseline_contra_scalers(metrics, pid, scope)
    if (not sc) or (not np.isfinite(sc["TBV"])) or (sc["TBV"] <= 0):
        continue

    for _, r in g.iterrows():
        o = dict(r)

        o["TBV_norm"] = r["TBV"] / sc["TBV"] if sc["TBV"] > 0 else np.nan

        for x in BVX_THRESHOLDS:
            bx_denom = sc.get(f"BV{x}", np.nan)
            o[f"BV{x}_norm"] = r[f"BV{x}"] / bx_denom if (np.isfinite(bx_denom) and bx_denom > 0) else np.nan

            frac_denom = sc.get(f"Frac_BV{x}", np.nan)
            o[f"Frac_BV{x}_norm"] = (r[f"Frac_BV{x}"] / frac_denom) if (np.isfinite(frac_denom) and frac_denom > 0) else np.nan

        norm_rows.append(o)

metrics_norm = pd.DataFrame(norm_rows)
assert not metrics_norm.empty, "Stage C: normalization produced no rows"


assert "is_in_tumor_lobe" in edges_df.columns, "edges_df missing is_in_tumor_lobe"
assert "lobe" in edges_df.columns, "edges_df missing lobe"


def _to_bool(x):
    s = str(x).strip().lower()
    if s in {"1","true","t","yes","y"}: return True
    if s in {"0","false","f","no","n"}: return False
    return np.nan

edges_df["is_in_tumor_lobe"] = edges_df["is_in_tumor_lobe"].apply(_to_bool)

bl_edges = edges_df[edges_df["tp_sel"] == "BL"].copy()

tumor_map_rows = []
for pid, g in bl_edges.groupby("patient"):
    gg = g[(g["is_in_tumor_lobe"] == True) & (g["lobe"].notna()) & (g["lobe"].astype(str) != "0")]
    if gg.empty:
        tumor_map_rows.append({"patient": pid, "tumor_lobe": None})
    else:
        tlobe = gg["lobe"].astype(str).mode().iloc[0]  # most common tumor-lobe label among tumor edges
        tumor_map_rows.append({"patient": pid, "tumor_lobe": tlobe})

tumor_lobe_map = pd.DataFrame(tumor_map_rows)


needed_tp_cols = {"patient","tp_sel","tp","tpraw","months","fu_order"}
missing = [c for c in needed_tp_cols if c not in edges_df.columns]
if missing:
    print("Warning: tp_selection missing cols in edges_df:", missing)

tp_cols = [c for c in ["patient","tp","tpraw","months","fu_order","tp_sel"] if c in edges_df.columns]
tp_selection = (
    edges_df.loc[edges_df["tp_sel"].isin(["BL","FU_earliest","FU_latest"]), tp_cols]
      .drop_duplicates()
      .sort_values(["patient","tp_sel"] + (["tpraw"] if "tpraw" in tp_cols else []))
)


metrics_norm.to_csv(OUT_DIR / "all_metrics_stageB_norm.csv", index=False)
tumor_lobe_map.to_csv(OUT_DIR / "tumor_lobe_map.csv", index=False)
tp_selection.to_csv(OUT_DIR / "tp_selection_stageB.csv", index=False)

print("Stage C complete")
#print(f" - metrics_norm rows: {len(metrics_norm):,}")
#print(" - tumor_lobe_map counts (None included):", tumor_lobe_map["tumor_lobe"].value_counts(dropna=False).to_dict())

display_head(metrics_norm.sort_values(["patient","tp_sel","scope","region"]), "metrics_norm (first rows)")
display_head(tumor_lobe_map, "tumor_lobe_map (first rows)")
display_head(tp_selection, "tp_selection (first rows)")


Stage C complete
metrics_norm (first rows)
-------------------------


,patient,tp_sel,scope,region,TBV,BV3,Frac_BV3,BV5,Frac_BV5,BV7,Frac_BV7,BV10,Frac_BV10,TBV_norm,BV3_norm,Frac_BV3_norm,BV5_norm,Frac_BV5_norm,BV7_norm,Frac_BV7_norm,BV10_norm,Frac_BV10_norm
3,P104,BL,A+V,central,220663.101638,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,66.739679,0.000302,1.048859,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000562,0.000536
2,P104,BL,A+V,contra,210383.955954,8468.036075,0.040250,52114.531998,0.247712,88401.360453,0.420191,118802.918964,0.564696,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
1,P104,BL,A+V,ipsi,204330.615178,7939.721448,0.038857,44047.538955,0.215570,76114.747495,0.372508,102837.981854,0.503292,0.971227,0.937611,0.965388,0.845206,0.870246,0.861013,0.886521,0.865618,0.891262
4,P104,BL,A+V,lobe=1,101851.622173,5173.051116,0.050790,26834.381483,0.263465,43798.400573,0.430022,57744.326291,0.566946,0.484123,0.610891,1.261853,0.514912,1.063598,0.495449,1.023397,0.486051,1.003984
5,P104,BL,A+V,lobe=2,102478.993005,2766.670332,0.026997,17213.157472,0.167968,32316.346922,0.315346,45093.655563,0.440028,0.487105,0.326719,0.670737,0.330295,0.678078,0.365564,0.750483,0.379567,0.779231
6,P104,BL,A+V,lobe=3,67979.926219,4367.358879,0.064245,21019.966161,0.309208,32857.047715,0.483335,42414.045907,0.623920,0.323123,0.515746,1.596129,0.403342,1.248260,0.371680,1.150275,0.357012,1.104878
7,P104,BL,A+V,lobe=4,33689.322617,1089.071289,0.032327,7326.762963,0.217480,14098.225293,0.418478,19275.320429,0.572149,0.160133,0.128610,0.803145,0.140590,0.877958,0.159480,0.995923,0.162246,1.013199
8,P104,BL,A+V,lobe=5,108714.707118,3011.605907,0.027702,23767.802874,0.218625,41446.087445,0.381237,57113.552628,0.525353,0.516744,0.355644,0.688240,0.456069,0.882581,0.468840,0.907296,0.480742,0.930329
0,P104,BL,A+V,whole,635377.672770,16407.757523,0.025824,96162.070953,0.151346,164516.107948,0.258926,221707.640498,0.348938,3.020086,1.937611,0.641575,1.845206,0.610978,1.861013,0.616212,1.866180,0.617923
30,P104,BL,Artery,central,110240.505124,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,66.739679,0.000605,0.991309,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000988,0.000997


tumor_lobe_map (first rows)
---------------------------


,patient,tumor_lobe
0,P104,2
1,P105,2
2,P106,1
3,P107,3
4,P109,3
5,P113,2
6,P124,5
7,P133,5
8,P135,5
9,P136,5


tp_selection (first rows)
-------------------------


,patient,tp,tpraw,months,fu_order,tp_sel
0,P104,BL,BASELINE-M0,0.0,0.0,BL
3945,P104,FU,FU1-M5.0,5.0,1.0,FU_earliest
16075,P104,FU,FU4-M41.0,41.0,4.0,FU_latest
34825,P105,BL,BASELINE-M0,0.0,0.0,BL
37921,P105,FU,FU1-M6.0,6.0,1.0,FU_earliest
48497,P105,FU,FU5-M40.0,40.0,5.0,FU_latest
63189,P106,BL,BASELINE-M0,0.0,0.0,BL
65903,P106,FU,FU1-M6.0,6.0,1.0,FU_earliest
68547,P106,FU,FU2-M11.0,11.0,2.0,FU_latest
78491,P107,BL,BASELINE-M0,0.0,0.0,BL


In [6]:
import numpy as np
import pandas as pd
import re


from pathlib import Path
import pandas as pd

BASE = Path(r"C:\Users\ilinc\OneDrive\Desktop\GraphAnalysis\GraphsCompleteAnalysis")
INPUT_DIR = BASE / "FinalStatisticalAnalysisResults"

TUMOR_MAP_PATH = INPUT_DIR / "tumor_lobe_map.csv"   

tumor_map_df = pd.read_csv(TUMOR_MAP_PATH)

def parse_lobe_set(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return set()
    s = str(x).strip()
    if s == "" or s.lower() in {"nan", "none"}:
        return set()
    parts = re.split(r"[,\;\|\s]+", s)
    out = {p.strip() for p in parts if p.strip()}
    out = {p for p in out if p in {"1","2","3","4","5"}}
    return out

tumor_map_df = pd.read_csv(TUMOR_MAP_PATH)
tumor_lobes = {str(r["patient"]): parse_lobe_set(r["tumor_lobe"]) for _, r in tumor_map_df.iterrows()}

def pick_weight_col(df):
    for c in ["volume_mm3", "vol_AL_mm3", "length"]:
        if c in df.columns:
            return c
    return None

def weighted_mean(x, w):
    x = pd.to_numeric(x, errors="coerce").to_numpy(dtype=float)
    w = pd.to_numeric(w, errors="coerce").to_numpy(dtype=float)
    m = np.isfinite(x) & np.isfinite(w) & (w > 0)
    if not np.any(m):
        return np.nan
    return float(np.sum(x[m] * w[m]) / np.sum(w[m]))

def compute_top2_table(edges_df, tree_name):
    df = edges_df.copy()

    if tree_name in {"Artery", "Vein"}:
        df = df[df["tree"] == tree_name]

    if "tp_sel" in df.columns:
        df = df[df["tp_sel"] == "BL"]

    df["lobe"] = df["lobe"].astype(str).str.strip()
    df = df[df["lobe"].isin(["1","2","3","4","5"])]

    df["dose_gy"] = pd.to_numeric(df["dose_gy"], errors="coerce")
    df = df[np.isfinite(df["dose_gy"])]

    wcol = pick_weight_col(df)
    if wcol is None:
        raise ValueError("No weight column found. Expected one of: volume_mm3, vol_AL_mm3, length")

    rows = []
    for (pid, lobe), g in df.groupby(["patient", "lobe"]):
        mu = weighted_mean(g["dose_gy"], g[wcol])   
        rows.append({"patient": str(pid), "lobe": str(lobe), "mean_dose_w": mu, "n_edges": len(g)})

    per_lobe = pd.DataFrame(rows)
    if per_lobe.empty:
        return pd.DataFrame()

    per_lobe = per_lobe.sort_values(["patient", "mean_dose_w"], ascending=[True, False])
    per_lobe["rank"] = per_lobe.groupby("patient")["mean_dose_w"].rank(method="first", ascending=False)

    out_rows = []
    for pid, g in per_lobe.groupby("patient"):
        g2 = g.dropna(subset=["mean_dose_w"]).sort_values("mean_dose_w", ascending=False)
        if len(g2) < 2:
            continue

        top2 = g2.head(2)["lobe"].tolist()
        top2_d = g2.head(2)["mean_dose_w"].tolist()

        tset = tumor_lobes.get(str(pid), set())
        in_top2 = (len(set(top2) & set(tset)) > 0) if tset else np.nan

        out_rows.append({
            "patient": str(pid),
            "tree": tree_name,
            "tumor_lobes": ",".join(sorted(tset)) if tset else "",
          #  "top1_lobe": top2[0],
           # "top2_lobe": top2[1],
            "tumor_in_top2": in_top2,
        })

    return pd.DataFrame(out_rows)

res_artery = compute_top2_table(edges_df, "Artery")
res_vein   = compute_top2_table(edges_df, "Vein")
#res_all    = compute_top2_table(edges_df, "All")  # uses both trees pooled (if present in edges_df)

def summarize(res, name):
    if res.empty:
        print(f"{name}: no results.")
        return
    valid = res["tumor_in_top2"].dropna()
    rate = float(valid.mean()) if len(valid) else np.nan
    print(f"{name}: N={len(res)} patients with >=2 lobes; tumor-in-top2 = {rate:.3f} (fraction of valid)")
    #display(res.sort_values(["tumor_in_top2","top1_mean_dose"], ascending=[True, False]).head(20))

summarize(res_artery, "Artery")
summarize(res_vein,   "Vein")
#summarize(res_all,    "All (pooled edges)")

combined = pd.concat([res_artery, res_vein], ignore_index=True)
# combined.to_csv(OUT_DIR / "tumor_lobe_in_top2_dose_by_tree.csv", index=False)
combined


Artery: N=30 patients with >=2 lobes; tumor-in-top2 = 1.000 (fraction of valid)
Vein: N=30 patients with >=2 lobes; tumor-in-top2 = 1.000 (fraction of valid)


,patient,tree,tumor_lobes,tumor_in_top2
0,P104,Artery,2,True
1,P105,Artery,2,True
2,P106,Artery,1,True
3,P107,Artery,3,True
4,P109,Artery,3,True
...,...,...,...,...
55,P76,Vein,2,True
56,P80,Vein,5,True
57,P82,Vein,2,True
58,P87,Vein,5,True


# BaselineCheck

- (A) Tumor side (ipsi) vs non-tumor side (contra) at BL
- (B) Tumor lobe vs median(other lobes) at BL


In [7]:
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown
from scipy import stats


BASE_DIR = Path(r"C:\Users\ilinc\OneDrive\Desktop\GraphAnalysis\GraphsCompleteAnalysis\FinalStatisticalAnalysisResults\PostResults")
METRICS_CSV   = BASE_DIR / "all_metrics_stageB_norm.csv"
TUMOR_MAP_CSV = BASE_DIR / "tumor_lobe_map.csv"
OUT_DIR = BASE_DIR / "FinalBaselineStats"
OUT_DIR.mkdir(parents=True, exist_ok=True)

assert METRICS_CSV.exists(), f"Missing: {METRICS_CSV}"


FIXED_INCLUDED_24 = [
    "P104","P105","P106","P107","P109","P113","P124","P133","P135","P136","P138",
    "P154","P165","P166","P180","P188","P54","P56","P60","P76","P80","P82","P87","P96"
]
FIXED_EXCLUDED_9 = ["P144","P151","P152","P155","P168","P172","P181","P183","P190"]
ALLOWED_PATIENTS = set(map(str, FIXED_INCLUDED_24))


SCOPES = ["Artery", "Vein"]  
BVX_THRESHOLDS = [3, 5, 7, 10]
METRICS_Q = ["TBV_norm", *(f"BV{x}_norm" for x in BVX_THRESHOLDS)]
ALPHA = 0.05
MIN_N = 5

USE_FDR = False

df = pd.read_csv(METRICS_CSV)
for c in ["patient", "scope", "region", "tp_sel"]:
    if c in df.columns:
        df[c] = df[c].astype(str)

df = df[df["patient"].isin(ALLOWED_PATIENTS)].copy()
df = df[df["scope"].isin(SCOPES)].copy()

df_bl = df[df["tp_sel"] == "BL"].copy()


def rbc_from_wilcoxon(diffs):
    d = np.asarray(diffs, float)
    d = d[np.isfinite(d)]
    d = d[d != 0]
    if d.size < MIN_N:
        return np.nan
    ranks = stats.rankdata(np.abs(d))
    w_pos = float(ranks[d > 0].sum())
    w_neg = float(ranks[d < 0].sum())
    denom = (w_pos + w_neg)
    return (w_pos - w_neg) / denom if denom > 0 else np.nan

def summarize_wilcoxon(diffs):
    d = np.asarray(diffs, float)
    d = d[np.isfinite(d)]
    d = d[d != 0]  
    n = int(d.size)
    if n < MIN_N:
        return dict(
            n=n, mean=np.nan, median=np.nan, q1=np.nan, q3=np.nan, sd=np.nan,
            p_raw=np.nan, test="N/A", eff=np.nan, eff_name="RBC"
        )
    mean = float(np.mean(d))
    med  = float(np.median(d))
    q1, q3 = float(np.quantile(d, 0.25)), float(np.quantile(d, 0.75))
    sd = float(np.std(d, ddof=1)) if n > 1 else np.nan

    stat, p = stats.wilcoxon(d, zero_method="wilcox", alternative="two-sided", correction=False, mode="auto")
    eff = rbc_from_wilcoxon(d)

    return dict(
        n=n, mean=mean, median=med, q1=q1, q3=q3, sd=sd,
        p_raw=float(p), test="Wilcoxon", eff=eff, eff_name="RBC"
    )

def fdr_bh(pvals):
    p = np.asarray(pvals, float)
    out = np.full_like(p, np.nan, dtype=float)
    msk = np.isfinite(p)
    pv = p[msk]
    if pv.size == 0:
        return out
    order = np.argsort(pv)
    ranked = pv[order]
    m = ranked.size
    adj = ranked * m / (np.arange(1, m + 1))
    adj = np.minimum.accumulate(adj[::-1])[::-1]
    adj = np.clip(adj, 0, 1)
    out_idx = np.where(msk)[0][order]
    out[out_idx] = adj
    return out

def _to_float(x):
    try:
        v = float(x)
        return v if np.isfinite(v) else np.nan
    except Exception:
        return np.nan

def _sig_mark(p):
    if pd.isna(p): return ""
    p = float(p)
    if p <= 0.001: return "***"
    if p <= 0.01:  return "**"
    if p <= 0.05:  return "*"
    return ""

def add_sig_markers(df, p_col):
    df = df.copy()
    df["sig"] = [_sig_mark(x) for x in df[p_col]]
    return df

def style_table(df, p_col, alpha=ALPHA):
    if df.empty:
        return df

    order = ["scope","metric","n","mean","median","q1","q3","sd","test","p_raw"]
    if p_col in df.columns and p_col != "p_raw":
        order.append(p_col)
    order += ["sig","eff_name","eff"]

    cols = [c for c in order if c in df.columns] + [c for c in df.columns if c not in order]
    df = df[cols].copy()

    if "p_raw" in df.columns:
        df["p_raw"] = pd.to_numeric(df["p_raw"], errors="coerce")
    if p_col in df.columns:
        df[p_col] = pd.to_numeric(df[p_col], errors="coerce")

    def bold_sig_rows(row):
        p = row.get(p_col, np.nan)
        if pd.notna(p) and float(p) <= alpha: 
            return ["font-weight: bold"] * len(row)
        return [""] * len(row)

    def _fmt_p(x):
        v = _to_float(x)
        if not np.isfinite(v): return ""
        if v < 0.001: return f"{v:.2e}"
        if v < 0.1:   return f"{v:.4f}"  
        return f"{v:.3f}"

    fmt = {
        "mean": "{:.4g}", "median": "{:.4g}", "q1": "{:.4g}", "q3": "{:.4g}",
        "sd": "{:.4g}", "eff": "{:.3g}"
    }
    if "p_raw" in df.columns:
        fmt["p_raw"] = _fmt_p
    if p_col in df.columns:
        fmt[p_col] = _fmt_p

    return (df.style
            .apply(bold_sig_rows, axis=1)
            .format(fmt))

def present_fixed_cohort():
    inc = ", ".join(FIXED_INCLUDED_24)
    exc = ", ".join(FIXED_EXCLUDED_9)
    display(Markdown(
        f"**Fixed cohort enforced globally (baseline stats)**  \n"
        f"- Included (N=24): `{inc}`  \n"
        f"- Excluded (N=9): `{exc}`  \n"
        f"- Scopes used: `{', '.join(SCOPES)}`  \n"
    ))

present_fixed_cohort()


rows_side = []
for scope in SCOPES:
    sub = df_bl[df_bl["scope"] == scope]
    if sub.empty:
        continue

    wide = sub.pivot_table(index=["patient"], columns="region", values=METRICS_Q, aggfunc="first")
    regions_lvl = set(wide.columns.get_level_values(1)) if hasattr(wide.columns, "get_level_values") else set()
    if not {"ipsi", "contra"}.issubset(regions_lvl):
        continue

    for met in METRICS_Q:
        try:
            ipsi   = pd.to_numeric(wide[(met, "ipsi")], errors="coerce")
            contra = pd.to_numeric(wide[(met, "contra")], errors="coerce")
        except KeyError:
            continue

        diffs = (ipsi - contra).dropna().to_numpy(float)
        res = summarize_wilcoxon(diffs)
        rows_side.append(dict(scope=scope, metric=met, **res))

tbl_side = pd.DataFrame(rows_side)

if not tbl_side.empty:
    if USE_FDR:
        tbl_side["p_fdr"] = fdr_bh(tbl_side["p_raw"].to_numpy(float))
        pcol = "p_fdr"
    else:
        pcol = "p_raw"

    tbl_side = add_sig_markers(tbl_side, p_col=pcol)

display(Markdown("## Baseline only — Tumor side (ipsi) vs non-tumor side (contra)"))
#display(Markdown("**Effect reported:** ipsi − contra (within-patient)"))
display(style_table(tbl_side, p_col=pcol))
out1 = OUT_DIR / ("BL_ipsi_vs_contra_NORM_noFrac.csv" if not USE_FDR else "BL_ipsi_vs_contra_NORM_noFrac_FDR.csv")
tbl_side.to_csv(out1, index=False)


tumor_map = {}
if TUMOR_MAP_CSV.exists():
    tm = pd.read_csv(TUMOR_MAP_CSV)
    if {"patient", "tumor_lobe"}.issubset(set(tm.columns)):
        tm["patient"] = tm["patient"].astype(str)
        tm = tm[tm["patient"].isin(ALLOWED_PATIENTS)].copy()
        tumor_map = {r["patient"]: (None if pd.isna(r["tumor_lobe"]) else str(r["tumor_lobe"])) for _, r in tm.iterrows()}

rows_lobe = []
for scope in SCOPES:
    sub = df_bl[(df_bl["scope"] == scope) & (df_bl["region"].str.startswith("lobe="))].copy()
    if sub.empty:
        continue

    for met in METRICS_Q:
        diffs = []
        for pid, g in sub.groupby("patient"):
            tl = tumor_map.get(pid, None)
            if not tl or tl == "0" or str(tl).lower() == "nan":
                continue

            gg = g[~g["region"].str.match(r"^lobe=0$")].copy()

            vals = {}
            for _, r in gg.iterrows():
                lobestr = str(r["region"]).split("=", 1)[1]
                v = pd.to_numeric(r.get(met, np.nan), errors="coerce")
                if np.isfinite(v):
                    vals[lobestr] = float(v)

            if tl not in vals:
                continue

            non = [v for k, v in vals.items() if k != tl and np.isfinite(v)]
            if len(non) == 0:
                continue

            diffs.append(vals[tl] - float(np.median(non)))

        if len(diffs) == 0:
            continue

        res = summarize_wilcoxon(diffs)
        rows_lobe.append(dict(scope=scope, metric=met, **res))

tbl_lobe = pd.DataFrame(rows_lobe)

if not tbl_lobe.empty:
    if USE_FDR:
        tbl_lobe["p_fdr"] = fdr_bh(tbl_lobe["p_raw"].to_numpy(float))
        pcol2 = "p_fdr"
    else:
        pcol2 = "p_raw"

    tbl_lobe = add_sig_markers(tbl_lobe, p_col=pcol2)

display(Markdown("## Baseline only — Tumor lobe vs median of other lobes"))
#display(Markdown("**Effect reported:** tumor_lobe − median(other lobes) (within-patient)"))
display(style_table(tbl_lobe, p_col=pcol2))
out2 = OUT_DIR / ("BL_tumor_lobe_vs_others_NORM_noFrac.csv" if not USE_FDR else "BL_tumor_lobe_vs_others_NORM_noFrac_FDR.csv")
tbl_lobe.to_csv(out2, index=False)


<frozen importlib._bootstrap>:228: RuntimeWarning: scipy._lib.messagestream.MessageStream size changed, may indicate binary incompatibility. Expected 56 from C header, got 64 from PyObject


**Fixed cohort enforced globally (baseline stats)**  
- Included (N=24): `P104, P105, P106, P107, P109, P113, P124, P133, P135, P136, P138, P154, P165, P166, P180, P188, P54, P56, P60, P76, P80, P82, P87, P96`  
- Excluded (N=9): `P144, P151, P152, P155, P168, P172, P181, P183, P190`  
- Scopes used: `Artery, Vein`  


## Baseline only — Tumor side (ipsi) vs non-tumor side (contra)

,scope,metric,n,mean,median,q1,q3,sd,test,p_raw,sig,eff_name,eff
0,Artery,TBV_norm,24,0.08208,0.02474,-0.0965,0.2467,0.3286,Wilcoxon,0.360,,RBC,0.22
1,Artery,BV3_norm,24,0.1342,0.04639,-0.2029,0.2334,0.4382,Wilcoxon,0.439,,RBC,0.187
2,Artery,BV5_norm,24,0.1023,0.06952,-0.153,0.302,0.3323,Wilcoxon,0.208,,RBC,0.3
3,Artery,BV7_norm,24,0.0908,0.1312,-0.1396,0.2297,0.3326,Wilcoxon,0.303,,RBC,0.247
4,Artery,BV10_norm,24,0.09675,0.08564,-0.1454,0.2441,0.3251,Wilcoxon,0.317,,RBC,0.24
5,Vein,TBV_norm,24,0.09453,0.09254,-0.09102,0.1942,0.2933,Wilcoxon,0.188,,RBC,0.313
6,Vein,BV3_norm,24,0.1267,0.06327,-0.2737,0.2675,0.5923,Wilcoxon,0.705,,RBC,0.0933
7,Vein,BV5_norm,24,0.08725,0.07862,-0.1813,0.3146,0.3299,Wilcoxon,0.303,,RBC,0.247
8,Vein,BV7_norm,24,0.08273,0.1154,-0.1578,0.2055,0.2979,Wilcoxon,0.290,,RBC,0.253
9,Vein,BV10_norm,24,0.08503,0.1008,-0.1386,0.2086,0.2961,Wilcoxon,0.241,,RBC,0.28


## Baseline only — Tumor lobe vs median of other lobes

,scope,metric,n,mean,median,q1,q3,sd,test,p_raw,sig,eff_name,eff
0,Artery,TBV_norm,24,0.1082,0.08915,0.02293,0.1416,0.1604,Wilcoxon,4.30e-04,***,RBC,0.773
1,Artery,BV3_norm,24,0.07966,0.05145,-0.04452,0.1792,0.1608,Wilcoxon,0.0395,*,RBC,0.48
2,Artery,BV5_norm,24,0.09084,0.03179,-0.01352,0.16,0.1372,Wilcoxon,0.0072,**,RBC,0.613
3,Artery,BV7_norm,24,0.08635,0.01471,-0.009739,0.1637,0.1456,Wilcoxon,0.0340,*,RBC,0.493
4,Artery,BV10_norm,24,0.09737,0.02404,-0.006321,0.166,0.1522,Wilcoxon,0.0053,**,RBC,0.633
5,Vein,TBV_norm,24,0.1201,0.102,0.02199,0.1619,0.138,Wilcoxon,4.42e-05,***,RBC,0.867
6,Vein,BV3_norm,24,0.09611,0.05884,-0.0221,0.1453,0.1768,Wilcoxon,0.0096,**,RBC,0.593
7,Vein,BV5_norm,24,0.08159,0.02209,-0.007052,0.1985,0.1261,Wilcoxon,0.0150,*,RBC,0.56
8,Vein,BV7_norm,24,0.08161,0.03413,0.003075,0.1253,0.1247,Wilcoxon,7.42e-04,***,RBC,0.747
9,Vein,BV10_norm,24,0.07798,0.03008,0.003255,0.1352,0.1176,Wilcoxon,0.0018,**,RBC,0.7


# Stage D — Statistical analysis 

In [8]:
import numpy as np
import pandas as pd
from scipy import stats
from IPython.display import display, Markdown
from pathlib import Path
import re


SCOPES = ["Artery", "Vein"]
ALPHA = 0.050
MIN_N = 5
BVX_THRESHOLDS = [3, 5, 7, 10]

RIGHT_LOBES = {"1", "2"}
LEFT_LOBES  = {"3", "4", "5"}

BASE = Path(r"C:\Users\ilinc\OneDrive\Desktop\GraphAnalysis\GraphsCompleteAnalysis")
RESULTS_DIR = BASE / "FinalStatisticalAnalysisResults" / "PostResults"
OUT_DIR = RESULTS_DIR / "FinalSampleStatsTest"
OUT_DIR.mkdir(parents=True, exist_ok=True)

EDGES_PATH        = RESULTS_DIR / "all_edges_tidy_stageA.csv"
METRICS_NORM_PATH = RESULTS_DIR / "all_metrics_stageB_norm.csv"
TUMOR_MAP_PATH    = RESULTS_DIR / "tumor_lobe_map.csv"   
assert EDGES_PATH.exists(), f"Missing: {EDGES_PATH}"
assert METRICS_NORM_PATH.exists(), f"Missing: {METRICS_NORM_PATH}"


FIXED_INCLUDED_24 = [
    "P104","P105","P106","P107","P109","P113","P124","P133","P135","P136","P138",
    "P154","P165","P166","P180","P188","P54","P56","P60","P76","P80","P82","P87","P96"
]
FIXED_EXCLUDED_9 = ["P144","P151","P152","P155","P168","P172","P181","P183","P190"]

ALLOWED_PATIENTS = set(map(str, FIXED_INCLUDED_24))

def cohen_dz(deltas):
    a = np.asarray(deltas, float); a = a[np.isfinite(a)]
    if a.size < 2: return np.nan
    sd = np.std(a, ddof=1)
    return float(np.mean(a) / sd) if sd > 0 else np.nan

def wilcoxon_r(deltas):
    a = np.asarray(deltas, float); a = a[np.isfinite(a)]
    n = a.size
    if n == 0: return np.nan
    pos = np.sum(a > 0); neg = np.sum(a < 0)
    return float((pos - neg) / n)

def choose_test_for_table(deltas_pooled):
    a = np.asarray(deltas_pooled, float); a = a[np.isfinite(a)]
    if a.size >= 15:
        try:
            sw_p = stats.shapiro(a).pvalue if 3 <= a.size <= 5000 else 0.0
        except Exception:
            sw_p = 0.0
        if sw_p > 0.05:
            return "ttest"
    return "wilcoxon"

def bootstrap_ci_mean(vec, nboot=5000, seed=123):
    vec = np.asarray(vec, float); vec = vec[np.isfinite(vec)]
    if vec.size < 2: return (np.nan, np.nan)
    rng = np.random.default_rng(seed); n = vec.size
    bs = np.array([np.mean(rng.choice(vec, size=n, replace=True)) for _ in range(nboot)])
    return float(np.quantile(bs, 0.025)), float(np.quantile(bs, 0.975))

def bootstrap_ci_median(vec, nboot=5000, seed=123):
    vec = np.asarray(vec, float); vec = vec[np.isfinite(vec)]
    if vec.size < 2: return (np.nan, np.nan)
    rng = np.random.default_rng(seed); n = vec.size
    bs = np.array([np.median(rng.choice(vec, size=n, replace=True)) for _ in range(nboot)])
    return float(np.quantile(bs, 0.025)), float(np.quantile(bs, 0.975))

def summarize_vector(vec, test_choice, min_n=MIN_N):
    vec = np.asarray(vec, float); vec = vec[np.isfinite(vec)]
    n = int(vec.size)
    if n < min_n:
        return dict(
            n=n, mean_delta=np.nan, median_delta=np.nan, q1=np.nan, q3=np.nan,
            sd=np.nan, ci_l=np.nan, ci_u=np.nan, ci_median_l=np.nan, ci_median_u=np.nan,
            test="N/A", p_raw=np.nan, eff=np.nan, eff_name=""
        )

    mean_delta = float(np.mean(vec)); median_delta = float(np.median(vec))
    q1 = float(np.quantile(vec, 0.25)); q3 = float(np.quantile(vec, 0.75))
    sd = float(np.std(vec, ddof=1)) if n > 1 else np.nan
    ci_l, ci_u = bootstrap_ci_mean(vec)
    ci_m_l, ci_m_u = bootstrap_ci_median(vec)

    if test_choice == "ttest":
        _, p = stats.ttest_1samp(vec, 0.0, nan_policy="omit")
        eff = cohen_dz(vec); eff_name = "dz"
        test_label = "paired t"
    else:
        try:
            _, p = stats.wilcoxon(vec, zero_method="pratt")
        except Exception:
            _, p = stats.wilcoxon(vec, zero_method="wilcox")
        eff = wilcoxon_r(vec); eff_name = "r"
        test_label = "Wilcoxon"

    return dict(
        n=n,
        mean_delta=mean_delta, median_delta=median_delta,
        q1=q1, q3=q3, sd=sd,
        ci_l=ci_l, ci_u=ci_u,
        ci_median_l=ci_m_l, ci_median_u=ci_m_u,
        test=test_label, p_raw=float(p),
        eff=float(eff) if np.isfinite(eff) else np.nan, eff_name=eff_name
    )


def deltas_by_patient(dfN, scope, region, metric, contrast, allowed_patients):
    fu = "FU_earliest" if contrast == "earliest" else "FU_latest"
    sub = dfN[(dfN.scope == scope) & (dfN.region == region) & (dfN.patient.isin(allowed_patients))]
    out = []
    for pid, g in sub.groupby("patient"):
        bl = g[g.tp_sel == "BL"]; fu_g = g[g.tp_sel == fu]
        if bl.empty or fu_g.empty:
            continue
        vb = pd.to_numeric(bl.iloc[0][metric], errors="coerce")
        vf = pd.to_numeric(fu_g.iloc[0][metric], errors="coerce")
        if np.isfinite(vb) and np.isfinite(vf):
            out.append(float(vf - vb))
    return np.array(out, float)

def tumor_side_deltas_from_lobes(dfN, scope, metric, contrast, allowed_patients, tumor_map):
    fu = "FU_earliest" if contrast == "earliest" else "FU_latest"
    lob = dfN[(dfN.scope == scope) & (dfN.region.str.startswith("lobe="))].copy()
    lob = lob[~lob.region.str.match(r"^lobe=0$")]
    lob = lob[lob.patient.isin(allowed_patients)]

    ipsi_d, contra_d, gap_d = [], [], []
    for pid in sorted(list(allowed_patients)):
        tl = tumor_map.get(pid, None)
        if (tl is None) or (str(tl) == "0") or (str(tl).lower() == "nan"):
            continue
        if str(tl) in RIGHT_LOBES:
            ipsi_set, contra_set = RIGHT_LOBES, LEFT_LOBES
        elif str(tl) in LEFT_LOBES:
            ipsi_set, contra_set = LEFT_LOBES, RIGHT_LOBES
        else:
            continue

        bl = lob[(lob.patient == pid) & (lob.tp_sel == "BL")]
        fu_g = lob[(lob.patient == pid) & (lob.tp_sel == fu)]
        if bl.empty or fu_g.empty:
            continue

        def side_sum(tp_df, side_set):
            vals = []
            for lr in tp_df.region.unique():
                lnum = lr.split("=", 1)[1]
                if lnum in side_set:
                    v = pd.to_numeric(tp_df.loc[tp_df.region == lr, metric].iloc[0], errors="coerce")
                    if np.isfinite(v):
                        vals.append(float(v))
            return float(np.sum(vals)) if len(vals) else np.nan

        ipsi_bl, contra_bl = side_sum(bl, ipsi_set), side_sum(bl, contra_set)
        ipsi_fu, contra_fu = side_sum(fu_g, ipsi_set), side_sum(fu_g, contra_set)
        if not (np.isfinite(ipsi_bl) and np.isfinite(contra_bl) and np.isfinite(ipsi_fu) and np.isfinite(contra_fu)):
            continue

        di = float(ipsi_fu - ipsi_bl)
        dc = float(contra_fu - contra_bl)
        ipsi_d.append(di); contra_d.append(dc); gap_d.append(di - dc)

    return np.array(ipsi_d, float), np.array(contra_d, float), np.array(gap_d, float)

def tumor_vs_non_gaps(dfN, tumor_map, scope, metric, contrast, allowed_patients):
    fu = "FU_earliest" if contrast == "earliest" else "FU_latest"
    sub = dfN[(dfN.scope == scope) & (dfN.region.str.startswith("lobe="))].copy()
    sub = sub[~sub.region.str.match(r"^lobe=0$")]
    sub = sub[sub.patient.isin(allowed_patients)]

    gaps = []
    for pid, g in sub.groupby("patient"):
        tl = tumor_map.get(pid)
        if tl is None or str(tl) == "0" or str(tl).lower() == "nan":
            continue
        bl = g[g.tp_sel == "BL"]; fu_g = g[g.tp_sel == fu]
        if bl.empty or fu_g.empty:
            continue

        lobes = sorted(list(set(bl.region) & set(fu_g.region)))
        if not lobes:
            continue

        deltas = {}
        for lr in lobes:
            lname = lr.split("=", 1)[1]
            vb = pd.to_numeric(bl.loc[bl.region == lr, metric].iloc[0], errors="coerce")
            vf = pd.to_numeric(fu_g.loc[fu_g.region == lr, metric].iloc[0], errors="coerce")
            if np.isfinite(vb) and np.isfinite(vf):
                deltas[lname] = float(vf - vb)

        if str(tl) not in deltas or len(deltas) <= 1:
            continue

        non_vals = [v for k, v in deltas.items() if k != str(tl)]
        if len(non_vals) == 0:
            continue

        gaps.append(deltas[str(tl)] - float(np.median(non_vals)))

    return np.array(gaps, float)


def _sig_mark(p):
    if pd.isna(p): return ""
    p = float(p)
    if p < 0.001: return "***"
    if p < 0.01:  return "**"
    if p < 0.05:  return "*"
    return ""

QUESTION_TEXT = {
    "Q1":  "Q1) Whole lung: ΔBVx_norm and ΔTBV_norm (FU − BL)",
    "Q3":  "Q3) Central region: ΔTBV_norm (FU − BL)",
    "Q4":  "Q4) Tumor-side vs contra (from tumor lobe): Δ (FU − BL) and gap (ipsi−contra)",
    "Q9":  "Q9) Tumor lobe vs non-tumor lobes: gap = Δtumor − median(Δnon)",
    "Q10": "Q10) Edge features: tumor lobe(s) vs non-tumor lobes (gap of normalized lobe-deltas)",
}

def add_sig_markers_raw(df):
    df = df.copy()
    if "p_raw" not in df.columns:
        df["p_raw"] = np.nan
    df["sig"] = [_sig_mark(pr) for pr in df["p_raw"]]
    return df

def style_table(df, p_col="p_raw", alpha=ALPHA):
    if df.empty:
        return df

    order = [
        "contrast","scope","region","metric",
        "n","mean_delta","median_delta","q1","q3","sd",
        "test","p_raw","sig","eff_name","eff"
    ]
    cols = [c for c in order if c in df.columns] + [c for c in df.columns if c not in order]
    df = df[cols].copy()

    def _to_float(x):
        try:
            v = float(x)
            return v if np.isfinite(v) else np.nan
        except Exception:
            return np.nan

    def _fmt_trunc(x, decimals=3):
        v = _to_float(x)
        if not np.isfinite(v):
            return ""
        factor = 10 ** decimals
        vt = np.trunc(v * factor) / factor
        return f"{vt:.{decimals}f}"

    if p_col in df.columns:
        df[p_col] = pd.to_numeric(df[p_col], errors="coerce")

    def bold_sig_rows(row):
        p = row.get(p_col, np.nan)
        if pd.notna(p) and float(p) < alpha:
            return ["font-weight: bold"] * len(row)
        return [""] * len(row)

    trunc3_cols = [c for c in ["mean_delta","median_delta","q1","q3","sd","eff"] if c in df.columns]
    fmt = {c: (lambda x, _f=_fmt_trunc: _f(x, 3)) for c in trunc3_cols}

    if "p_raw" in df.columns:
        fmt["p_raw"] = lambda x: ("" if not np.isfinite(_to_float(x)) else f"{_to_float(x):.3g}")

    return df.style.apply(bold_sig_rows, axis=1).format(fmt)

def present_table_raw(qname, contrast, df):
    df2 = add_sig_markers_raw(df)
    display(Markdown(f"### {QUESTION_TEXT[qname]}  \n**Contrast:** {contrast.upper()}"))
    display(style_table(df2, p_col="p_raw", alpha=ALPHA))

def present_fixed_cohort():
    inc = ", ".join(FIXED_INCLUDED_24)
    exc = ", ".join(FIXED_EXCLUDED_9)
    display(Markdown(
        f"**Fixed cohort enforced globally**  \n"
        f"- Included (N=24): `{inc}`  \n"
        f"- Excluded (N=9): `{exc}`"
    ))


POSSIBLE_FEATURES = {
    "length":        ("sum",    None),
    "tortuosity":    ("wmean",  "length"),
    "radius_avg":    ("wmean",  "vol_AL_mm3"),
    "CSA_mm2":       ("wmean",  "vol_AL_mm3"),
    "surface_area":  ("sum",    None),
    "volume_mm3":    ("sum",    None),
}

def _weighted_mean(x, w):
    x = np.asarray(x, float); w = np.asarray(w, float)
    m = np.isfinite(x) & np.isfinite(w) & (w > 0)
    if not m.any(): return np.nan
    sw = w[m].sum()
    return float((x[m] @ w[m]) / sw) if sw > 0 else np.nan

def aggregate_edge_features(df, by_cols, EDGE_FEATURES):
    rows = []
    for key, g in df.groupby(by_cols, dropna=False):
        rec = dict(zip(by_cols, key if isinstance(key, tuple) else (key,)))
        for f in EDGE_FEATURES:
            agg, wcol = POSSIBLE_FEATURES[f]
            if agg == "sum":
                rec[f] = float(np.nansum(pd.to_numeric(g[f], errors="coerce").values))
            else:
                x = pd.to_numeric(g[f], errors="coerce")
                if wcol is not None and wcol in g.columns:
                    w = pd.to_numeric(g[wcol], errors="coerce")
                    rec[f] = float(_weighted_mean(x, w))
                else:
                    rec[f] = float(np.nanmean(x.values))
        rows.append(rec)
    return pd.DataFrame(rows)

def add_AplusV_edges(df, by_cols_base, EDGE_FEATURES):
    if df.empty or "scope" not in df.columns:
        return df
    if not {"Artery","Vein"}.issubset(set(df["scope"].unique())):
        return df
    av = (df[df["scope"].isin(["Artery","Vein"])]
          .groupby(by_cols_base, as_index=False)[EDGE_FEATURES].mean())
    av["scope"] = "A+V"
    cols = by_cols_base + ["scope"] + EDGE_FEATURES
    return pd.concat([df, av[cols]], ignore_index=True)

def build_anchors(edges_df, EDGE_FEATURES):
    eb = edges_df[edges_df["tp_sel"] == "BL"].copy()
    eb = eb[np.isfinite(pd.to_numeric(eb["vol_AL_mm3"], errors="coerce"))]
    if "region" in eb.columns:
        eb["region2"] = np.where(eb["region"].isin(["ipsi","contra"]), eb["region"], "whole")
        bl_side = aggregate_edge_features(eb, ["patient","scope","region2"], EDGE_FEATURES).rename(columns={"region2":"region"})
        bl_side = add_AplusV_edges(bl_side, ["patient","region"], EDGE_FEATURES)
        contra = bl_side[bl_side["region"] == "contra"].drop(columns=["region"], errors="ignore").copy()
        whole = bl_side.groupby(["patient","scope"], as_index=False)[EDGE_FEATURES].sum()
        contra_keys = set(zip(contra["patient"], contra["scope"]))
        whole_fallback = whole[[((p,s) not in contra_keys) for p,s in zip(whole["patient"], whole["scope"])]].copy()
        anchors = pd.concat([contra, whole_fallback], ignore_index=True)
    else:
        whole = aggregate_edge_features(eb, ["patient","scope"], EDGE_FEATURES)
        anchors = add_AplusV_edges(whole, ["patient"], EDGE_FEATURES)
    anchors = anchors.drop_duplicates(subset=["patient","scope"], keep="first")
    return anchors

def normalize_block(df_agg, anchors_df, on_cols, EDGE_FEATURES):
    m = df_agg.merge(anchors_df, on=["patient","scope"], suffixes=("", "_ANCH"), how="left")
    out = m[on_cols].copy()
    for f in EDGE_FEATURES:
        denom = pd.to_numeric(m[f + "_ANCH"], errors="coerce")
        num   = pd.to_numeric(m[f], errors="coerce")
        denom = denom.where(np.isfinite(denom) & (denom != 0), np.nan)
        out[f] = num / denom
    return out

def deltas_by_lobe_norm(edges_df, anchors_df, contrast, EDGE_FEATURES, allowed_patients):
    fu = "FU_earliest" if contrast == "earliest" else "FU_latest"
    sub = edges_df[edges_df["patient"].isin(allowed_patients)].copy()
    sub = sub[sub["lobe"].astype(str) != "0"].copy()
    sub = sub[np.isfinite(pd.to_numeric(sub["vol_AL_mm3"], errors="coerce"))]

    bl_agg = aggregate_edge_features(sub[sub["tp_sel"] == "BL"], ["patient","scope","lobe"], EDGE_FEATURES)
    fu_agg = aggregate_edge_features(sub[sub["tp_sel"] == fu], ["patient","scope","lobe"], EDGE_FEATURES)

    bl_agg = add_AplusV_edges(bl_agg, ["patient","lobe"], EDGE_FEATURES)
    fu_agg = add_AplusV_edges(fu_agg, ["patient","lobe"], EDGE_FEATURES)

    bl_n = normalize_block(bl_agg, anchors_df, ["patient","scope","lobe"], EDGE_FEATURES)
    fu_n = normalize_block(fu_agg, anchors_df, ["patient","scope","lobe"], EDGE_FEATURES)

    merged = bl_n.merge(fu_n, on=["patient","scope","lobe"], suffixes=("_BL","_FU"), how="inner")
    if merged.empty:
        return pd.DataFrame()

    out = merged[["patient","scope","lobe"]].copy()
    for f in EDGE_FEATURES:
        out[f] = pd.to_numeric(merged[f"{f}_FU"], errors="coerce") - pd.to_numeric(merged[f"{f}_BL"], errors="coerce")
    return out

def run_table_Q10_edges(edges_df, anchors_df, contrast, tumor_map, EDGE_FEATURES, allowed_patients):
    d_lobe = deltas_by_lobe_norm(edges_df, anchors_df, contrast, EDGE_FEATURES, allowed_patients)
    if d_lobe.empty:
        return pd.DataFrame()

    pooled = []
    dA = d_lobe[d_lobe["scope"] == "A+V"].copy()

    for feat in EDGE_FEATURES:
        for pid, g in dA.groupby("patient"):
            tset = tumor_map.get(pid, set())
            m = g[["lobe", feat]].copy()
            m["lobe"] = m["lobe"].astype(str)
            m[feat] = pd.to_numeric(m[feat], errors="coerce")
            m = m[np.isfinite(m[feat].values)]
            if m["lobe"].nunique() < 2:
                continue
            in_set = m["lobe"].isin(tset)
            if (not in_set.any()) or ((~in_set).sum() == 0):
                continue
            tumor_val = float(np.median(m.loc[in_set, feat]))
            non_vals = m.loc[~in_set, feat].to_numpy(dtype=float)
            pooled.append(tumor_val - float(np.median(non_vals)))

    test = choose_test_for_table(pooled)

    rows = []
    for scope in [s for s in SCOPES if s in d_lobe["scope"].unique()]:
        ds = d_lobe[d_lobe["scope"] == scope].copy()
        for feat in EDGE_FEATURES:
            gaps = []
            for pid, g in ds.groupby("patient"):
                tset = tumor_map.get(pid, set())
                m = g[["lobe", feat]].copy()
                m["lobe"] = m["lobe"].astype(str)
                m[feat] = pd.to_numeric(m[feat], errors="coerce")
                m = m[np.isfinite(m[feat].values)]
                if m["lobe"].nunique() < 2:
                    continue
                in_set = m["lobe"].isin(tset)
                if (not in_set.any()) or ((~in_set).sum() == 0):
                    continue
                tumor_val = float(np.median(m.loc[in_set, feat]))
                non_vals = m.loc[~in_set, feat].to_numpy(dtype=float)
                gaps.append(tumor_val - float(np.median(non_vals)))

            rows.append(dict(
                contrast=contrast, scope=scope, region="tumor_vs_non_median_gap",
                metric=feat, **summarize_vector(np.asarray(gaps), test)
            ))

    return pd.DataFrame(rows)


edges = pd.read_csv(EDGES_PATH)
metrics_norm = pd.read_csv(METRICS_NORM_PATH)

if "scope" not in edges.columns and "tree" in edges.columns:
    edges = edges.rename(columns={"tree": "scope"})
if "scope" not in metrics_norm.columns and "tree" in metrics_norm.columns:
    metrics_norm = metrics_norm.rename(columns={"tree": "scope"})

edges["patient"] = edges["patient"].astype(str)
metrics_norm["patient"] = metrics_norm["patient"].astype(str)

if "tp_sel" not in edges.columns:
    raise ValueError("all_edges_tidy_stageA.csv must include 'tp_sel'.")
if "tp_sel" not in metrics_norm.columns:
    raise ValueError("all_metrics_stageB_norm.csv must include 'tp_sel'.")

edges["tp_sel"] = edges["tp_sel"].astype(str)
metrics_norm["tp_sel"] = metrics_norm["tp_sel"].astype(str)

edges = edges[edges["patient"].isin(ALLOWED_PATIENTS)].copy()
metrics_norm = metrics_norm[metrics_norm["patient"].isin(ALLOWED_PATIENTS)].copy()

if TUMOR_MAP_PATH.exists():
    tumor_lobe_map = pd.read_csv(TUMOR_MAP_PATH)
else:
    tumor_rows = []
    ebl = edges[edges["tp_sel"] == "BL"].copy()
    if "lobe" not in ebl.columns:
        raise ValueError("Cannot rebuild tumor_lobe_map: edges missing 'lobe'")
    if "is_in_tumor_lobe" not in ebl.columns:
        raise ValueError("Cannot rebuild tumor_lobe_map: edges missing 'is_in_tumor_lobe'")

    ebl["lobe"] = ebl["lobe"].astype(str)

    def _to_bool(x):
        s = str(x).strip().lower()
        if s in {"1","true","t","yes","y"}: return True
        if s in {"0","false","f","no","n"}: return False
        return np.nan

    ebl["is_in_tumor_lobe"] = ebl["is_in_tumor_lobe"].apply(_to_bool)

    for pid, g in ebl.groupby("patient"):
        gg = g[(g["is_in_tumor_lobe"] == True) & (g["lobe"].notna()) & (g["lobe"].astype(str) != "0")]
        if gg.empty:
            tumor_rows.append({"patient": pid, "tumor_lobe": None})
        else:
            tumor_rows.append({"patient": pid, "tumor_lobe": gg["lobe"].astype(str).mode().iloc[0]})

    tumor_lobe_map = pd.DataFrame(tumor_rows)
    tumor_lobe_map.to_csv(TUMOR_MAP_PATH, index=False)

tumor_lobe_map["patient"] = tumor_lobe_map["patient"].astype(str)
tumor_lobe_map = tumor_lobe_map[tumor_lobe_map["patient"].isin(ALLOWED_PATIENTS)].copy()

tumap_dict = {r["patient"]: (None if pd.isna(r["tumor_lobe"]) else str(r["tumor_lobe"])) for _, r in tumor_lobe_map.iterrows()}

def _parse_lobe_set(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return set()
    s = str(x).strip()
    if s == "" or s.lower() in {"nan", "none"}:
        return set()
    parts = re.split(r"[,\;\|\s]+", s)
    return {p.strip() for p in parts if p.strip()}

tumap_edge = {str(r["patient"]): _parse_lobe_set(r["tumor_lobe"]) for _, r in tumor_lobe_map.iterrows()} if not tumor_lobe_map.empty else {}

if "lobe" not in edges.columns:
    raise ValueError("all_edges_tidy_stageA.csv must include 'lobe'.")
edges["lobe"] = edges["lobe"].astype(str)

if "vol_AL_mm3" not in edges.columns:
    raise ValueError("Need 'vol_AL_mm3' in all_edges_tidy_stageA.csv.")

EDGE_FEATURES = [f for f in POSSIBLE_FEATURES if f in edges.columns]
if not EDGE_FEATURES:
    raise ValueError("No supported edge features found in all_edges_tidy_stageA.csv.")

anchors = build_anchors(edges, EDGE_FEATURES)


def run_table_Q1(dfN, contrast, allowed_patients):
    mets = [*(f"BV{x}_norm" for x in BVX_THRESHOLDS), "TBV_norm"]
    pooled = []
    for met in mets:
        pooled.extend(deltas_by_patient(dfN, "A+V", "whole", met, contrast, allowed_patients))
    test = choose_test_for_table(pooled)

    rows = []
    for scope in SCOPES:
        for met in mets:
            d = deltas_by_patient(dfN, scope, "whole", met, contrast, allowed_patients)
            rows.append(dict(contrast=contrast, scope=scope, region="whole", metric=met, **summarize_vector(d, test)))
    return pd.DataFrame(rows)

def run_table_Q3(dfN, contrast, allowed_patients):
    pooled = deltas_by_patient(dfN, "A+V", "central", "TBV_norm", contrast, allowed_patients)
    test = choose_test_for_table(pooled)

    rows = []
    for scope in SCOPES:
        d = deltas_by_patient(dfN, scope, "central", "TBV_norm", contrast, allowed_patients)
        rows.append(dict(contrast=contrast, scope=scope, region="central", metric="TBV_norm", **summarize_vector(d, test)))
    return pd.DataFrame(rows)

def run_table_Q4(dfN, contrast, allowed_patients, tumor_map):
    mets = ["TBV_norm", *(f"BV{x}_norm" for x in BVX_THRESHOLDS)]

    pooled = []
    for met in mets:
        di, dc, gap = tumor_side_deltas_from_lobes(dfN, "A+V", met, contrast, allowed_patients, tumor_map)
        pooled.extend(di); pooled.extend(dc); pooled.extend(gap)
    test = choose_test_for_table(pooled)

    rows = []
    for scope in SCOPES:
        for met in mets:
            di, dc, gap = tumor_side_deltas_from_lobes(dfN, scope, met, contrast, allowed_patients, tumor_map)
            rows.append(dict(contrast=contrast, scope=scope, region="ipsi(tumor-side)", metric=met, **summarize_vector(di, test)))
            rows.append(dict(contrast=contrast, scope=scope, region="contra(opposite-side)", metric=met, **summarize_vector(dc, test)))
            rows.append(dict(contrast=contrast, scope=scope, region="gap_ipsi_minus_contra", metric=met, **summarize_vector(gap, test)))
    return pd.DataFrame(rows)

def run_table_Q9(dfN, contrast, allowed_patients, tumor_map):
    mets = ["TBV_norm", *(f"BV{x}_norm" for x in BVX_THRESHOLDS)]

    pooled = []
    for met in mets:
        pooled.extend(tumor_vs_non_gaps(dfN, tumor_map, "A+V", met, contrast, allowed_patients))
    test = choose_test_for_table(pooled)

    rows = []
    for scope in SCOPES:
        for met in mets:
            gaps = tumor_vs_non_gaps(dfN, tumor_map, scope, met, contrast, allowed_patients)
            rows.append(dict(contrast=contrast, scope=scope, region="tumor_vs_non_median_gap", metric=met, **summarize_vector(gaps, test)))
    return pd.DataFrame(rows)


present_fixed_cohort()

SAVED = []

# Q1
dfs = []
for contrast in ["earliest", "latest"]:
    dfs.append(run_table_Q1(metrics_norm, contrast, ALLOWED_PATIENTS))
df_Q1 = pd.concat(dfs, ignore_index=True)
present_table_raw("Q1", "earliest+latest", df_Q1)
out = OUT_DIR / "Q1_all.csv"; df_Q1.to_csv(out, index=False); SAVED.append(out)

# Q3
dfs = []
for contrast in ["earliest", "latest"]:
    dfs.append(run_table_Q3(metrics_norm, contrast, ALLOWED_PATIENTS))
df_Q3 = pd.concat(dfs, ignore_index=True)
present_table_raw("Q3", "earliest+latest", df_Q3)
out = OUT_DIR / "Q3_all.csv"; df_Q3.to_csv(out, index=False); SAVED.append(out)

# Q4
dfs = []
for contrast in ["earliest", "latest"]:
    dfs.append(run_table_Q4(metrics_norm, contrast, ALLOWED_PATIENTS, tumap_dict))
df_Q4 = pd.concat(dfs, ignore_index=True)
present_table_raw("Q4", "earliest+latest", df_Q4)
out = OUT_DIR / "Q4_all.csv"; df_Q4.to_csv(out, index=False); SAVED.append(out)

# Q9
dfs = []
for contrast in ["earliest", "latest"]:
    dfs.append(run_table_Q9(metrics_norm, contrast, ALLOWED_PATIENTS, tumap_dict))
df_Q9 = pd.concat(dfs, ignore_index=True)
present_table_raw("Q9", "earliest+latest", df_Q9)
out = OUT_DIR / "Q9_all.csv"; df_Q9.to_csv(out, index=False); SAVED.append(out)

# Q10
dfs = []
for contrast in ["earliest", "latest"]:
    dfs.append(run_table_Q10_edges(edges, anchors, contrast, tumap_edge, EDGE_FEATURES, ALLOWED_PATIENTS))
df_Q10 = pd.concat(dfs, ignore_index=True)
present_table_raw("Q10", "earliest+latest", df_Q10)
out = OUT_DIR / "Q10_edge_tumor_vs_non_all.csv"; df_Q10.to_csv(out, index=False); SAVED.append(out)




**Fixed cohort enforced globally**  
- Included (N=24): `P104, P105, P106, P107, P109, P113, P124, P133, P135, P136, P138, P154, P165, P166, P180, P188, P54, P56, P60, P76, P80, P82, P87, P96`  
- Excluded (N=9): `P144, P151, P152, P155, P168, P172, P181, P183, P190`

### Q1) Whole lung: ΔBVx_norm and ΔTBV_norm (FU − BL)  
**Contrast:** EARLIEST+LATEST

,contrast,scope,region,metric,n,mean_delta,median_delta,q1,q3,sd,test,p_raw,sig,eff_name,eff,ci_l,ci_u,ci_median_l,ci_median_u
0,earliest,Artery,whole,BV3_norm,24,1.039,0.185,-0.657,1.832,3.032,Wilcoxon,0.197,,r,0.083,0.037799,2.388595,-0.520062,1.358848
1,earliest,Artery,whole,BV5_norm,24,0.144,0.029,-0.321,0.229,0.781,Wilcoxon,0.855,,r,0.083,-0.148500,0.459601,-0.275818,0.196615
2,earliest,Artery,whole,BV7_norm,24,0.095,-0.017,-0.271,0.450,0.542,Wilcoxon,0.684,,r,-0.083,-0.107551,0.307389,-0.198508,0.400000
3,earliest,Artery,whole,BV10_norm,24,0.040,-0.013,-0.249,0.367,0.484,Wilcoxon,0.79,,r,-0.083,-0.143193,0.231023,-0.150187,0.240020
4,earliest,Artery,whole,TBV_norm,24,-0.093,-0.015,-0.355,0.071,0.515,Wilcoxon,0.473,,r,-0.083,-0.297782,0.104179,-0.105318,0.054801
5,earliest,Vein,whole,BV3_norm,24,1.586,0.274,-0.466,1.853,4.335,Wilcoxon,0.143,,r,0.166,0.192318,3.508647,-0.151960,1.240232
6,earliest,Vein,whole,BV5_norm,24,0.322,0.073,-0.197,0.606,1.120,Wilcoxon,0.439,,r,0.083,-0.058985,0.796254,-0.144881,0.422980
7,earliest,Vein,whole,BV7_norm,24,0.094,0.009,-0.153,0.127,0.450,Wilcoxon,0.747,,r,0.083,-0.062609,0.287364,-0.124530,0.078309
8,earliest,Vein,whole,BV10_norm,24,0.069,0.052,-0.147,0.203,0.389,Wilcoxon,0.565,,r,0.250,-0.076390,0.232773,-0.118132,0.152530
9,earliest,Vein,whole,TBV_norm,24,-0.161,-0.087,-0.271,0.054,0.369,Wilcoxon,0.101,,r,-0.333,-0.313687,-0.022115,-0.209337,0.031709


### Q3) Central region: ΔTBV_norm (FU − BL)  
**Contrast:** EARLIEST+LATEST

,contrast,scope,region,metric,n,mean_delta,median_delta,q1,q3,sd,test,p_raw,sig,eff_name,eff,ci_l,ci_u,ci_median_l,ci_median_u
0,earliest,Artery,central,TBV_norm,24,-0.158,-0.005,-0.112,0.110,0.532,Wilcoxon,0.79,,r,-0.083,-0.388003,0.023390,-0.079048,0.076604
1,earliest,Vein,central,TBV_norm,24,-0.222,-0.111,-0.273,0.085,0.567,Wilcoxon,0.0646,,r,-0.250,-0.464259,-0.027579,-0.263937,0.067575
2,latest,Artery,central,TBV_norm,24,-0.138,0.096,-0.147,0.179,0.591,Wilcoxon,0.726,,r,0.333,-0.391204,0.067776,-0.062883,0.151916
3,latest,Vein,central,TBV_norm,24,-0.148,0.025,-0.331,0.180,0.552,Wilcoxon,0.747,,r,0.083,-0.380388,0.049306,-0.114829,0.159703


...

,contrast,scope,region,metric,n,mean_delta,median_delta,q1,q3,sd,test,p_raw,sig,eff_name,eff,ci_l,ci_u,ci_median_l,ci_median_u
0,earliest,Artery,ipsi(tumor-side),TBV_norm,24,0.025,-0.022,-0.174,0.046,0.312,Wilcoxon,0.509,,r,-0.333,-0.087052,0.151630,-0.103899,0.017513
1,earliest,Artery,contra(opposite-side),TBV_norm,24,0.039,0.028,-0.047,0.117,0.276,Wilcoxon,0.406,,r,0.083,-0.067103,0.145594,-0.023526,0.097180
2,earliest,Artery,gap_ipsi_minus_contra,TBV_norm,24,-0.013,-0.035,-0.077,0.053,0.120,Wilcoxon,0.456,,r,-0.250,-0.061053,0.033963,-0.068701,0.046651
3,earliest,Artery,ipsi(tumor-side),BV3_norm,24,0.509,0.152,-0.266,0.794,1.478,Wilcoxon,0.208,,r,0.166,-0.005227,1.151953,-0.153516,0.629376
4,earliest,Artery,contra(opposite-side),BV3_norm,24,0.544,0.150,-0.322,0.790,1.563,Wilcoxon,0.143,,r,0.083,0.045317,1.233476,-0.275988,0.729472
5,earliest,Artery,gap_ipsi_minus_contra,BV3_norm,24,-0.035,-0.017,-0.184,0.151,0.468,Wilcoxon,0.684,,r,-0.166,-0.223281,0.151134,-0.166447,0.097927
6,earliest,Artery,ipsi(tumor-side),BV5_norm,24,0.075,-0.009,-0.166,0.206,0.429,Wilcoxon,0.768,,r,0.000,-0.084475,0.251623,-0.135104,0.164342
7,earliest,Artery,contra(opposite-side),BV5_norm,24,0.098,0.036,-0.126,0.172,0.383,Wilcoxon,0.422,,r,0.166,-0.046065,0.251284,-0.092032,0.156623
8,earliest,Artery,gap_ipsi_minus_contra,BV5_norm,24,-0.022,0.011,-0.136,0.067,0.188,Wilcoxon,0.406,,r,0.083,-0.093306,0.057728,-0.113699,0.044327
9,earliest,Artery,ipsi(tumor-side),BV7_norm,24,0.050,-0.018,-0.169,0.230,0.296,Wilcoxon,0.747,,r,-0.083,-0.058429,0.167802,-0.086987,0.149700


### Q9) Tumor lobe vs non-tumor lobes: gap = Δtumor − median(Δnon)  
**Contrast:** EARLIEST+LATEST

,contrast,scope,region,metric,n,mean_delta,median_delta,q1,q3,sd,test,p_raw,sig,eff_name,eff,ci_l,ci_u,ci_median_l,ci_median_u
0,earliest,Artery,tumor_vs_non_median_gap,TBV_norm,24,-0.056,-0.070,-0.109,-0.038,0.096,Wilcoxon,0.00533,**,r,-0.750,-0.091844,-0.017270,-0.104832,-0.046582
1,earliest,Artery,tumor_vs_non_median_gap,BV3_norm,24,-0.042,-0.025,-0.125,0.040,0.163,Wilcoxon,0.375,,r,-0.083,-0.105504,0.016862,-0.098605,0.020755
2,earliest,Artery,tumor_vs_non_median_gap,BV5_norm,24,-0.074,-0.062,-0.127,-0.025,0.127,Wilcoxon,0.00533,**,r,-0.583,-0.124841,-0.026445,-0.109833,-0.036807
3,earliest,Artery,tumor_vs_non_median_gap,BV7_norm,24,-0.064,-0.059,-0.099,-0.019,0.103,Wilcoxon,0.0059,**,r,-0.583,-0.104454,-0.025602,-0.093150,-0.027859
4,earliest,Artery,tumor_vs_non_median_gap,BV10_norm,24,-0.067,-0.075,-0.107,-0.012,0.104,Wilcoxon,0.00434,**,r,-0.583,-0.107932,-0.027536,-0.089927,-0.029850
5,earliest,Vein,tumor_vs_non_median_gap,TBV_norm,24,-0.044,-0.061,-0.081,0.010,0.100,Wilcoxon,0.0115,*,r,-0.416,-0.083216,-0.004386,-0.081363,-0.016895
6,earliest,Vein,tumor_vs_non_median_gap,BV3_norm,24,-0.067,-0.047,-0.111,0.051,0.181,Wilcoxon,0.101,,r,-0.333,-0.142846,-0.002985,-0.070981,0.015524
7,earliest,Vein,tumor_vs_non_median_gap,BV5_norm,24,-0.046,-0.028,-0.096,0.038,0.133,Wilcoxon,0.16,,r,-0.250,-0.098884,0.004892,-0.072587,0.021020
8,earliest,Vein,tumor_vs_non_median_gap,BV7_norm,24,-0.051,-0.040,-0.128,0.015,0.103,Wilcoxon,0.0395,*,r,-0.333,-0.091546,-0.011268,-0.123305,0.005624
9,earliest,Vein,tumor_vs_non_median_gap,BV10_norm,24,-0.044,-0.030,-0.105,0.026,0.098,Wilcoxon,0.0425,*,r,-0.416,-0.082859,-0.006455,-0.093441,0.002830


### Q10) Edge features: tumor lobe(s) vs non-tumor lobes (gap of normalized lobe-deltas)  
**Contrast:** EARLIEST+LATEST

,contrast,scope,region,metric,n,mean_delta,median_delta,q1,q3,sd,test,p_raw,sig,eff_name,eff,ci_l,ci_u,ci_median_l,ci_median_u
0,earliest,Artery,tumor_vs_non_median_gap,length,24,-0.057,-0.076,-0.098,-0.018,0.095,Wilcoxon,0.00792,**,r,-0.583,-0.094429,-0.019776,-0.097309,-0.022467
1,earliest,Artery,tumor_vs_non_median_gap,tortuosity,24,0.002,0.001,-0.005,0.012,0.015,Wilcoxon,0.345,,r,0.000,-0.003760,0.008782,-0.003709,0.008349
2,earliest,Artery,tumor_vs_non_median_gap,radius_avg,24,-0.000,0.025,-0.051,0.061,0.101,Wilcoxon,0.877,,r,0.083,-0.041359,0.037918,-0.035302,0.055844
3,earliest,Artery,tumor_vs_non_median_gap,CSA_mm2,24,-0.010,0.055,-0.110,0.138,0.270,Wilcoxon,0.9,,r,0.083,-0.122094,0.090397,-0.079757,0.123374
4,earliest,Artery,tumor_vs_non_median_gap,surface_area,24,-0.057,-0.077,-0.104,-0.022,0.091,Wilcoxon,0.00792,**,r,-0.666,-0.091761,-0.020770,-0.094310,-0.032588
5,earliest,Artery,tumor_vs_non_median_gap,volume_mm3,24,-0.056,-0.070,-0.109,-0.038,0.096,Wilcoxon,0.00533,**,r,-0.750,-0.091844,-0.017270,-0.104832,-0.046582
6,earliest,Vein,tumor_vs_non_median_gap,length,24,-0.045,-0.039,-0.096,0.001,0.090,Wilcoxon,0.0164,*,r,-0.500,-0.081108,-0.010250,-0.080501,-0.020239
7,earliest,Vein,tumor_vs_non_median_gap,tortuosity,24,0.009,0.010,-0.001,0.015,0.032,Wilcoxon,0.0138,*,r,0.333,-0.003305,0.022070,0.000243,0.015485
8,earliest,Vein,tumor_vs_non_median_gap,radius_avg,24,-0.013,-0.012,-0.065,0.030,0.086,Wilcoxon,0.584,,r,-0.083,-0.049125,0.019613,-0.054587,0.020921
9,earliest,Vein,tumor_vs_non_median_gap,CSA_mm2,24,-0.038,-0.017,-0.179,0.076,0.225,Wilcoxon,0.406,,r,-0.083,-0.128194,0.049396,-0.153817,0.038515
